In [ ]:
# Mount Google Drive and initialize project paths

from google.colab import drive
from pathlib import Path

import json
import random
import time
import hashlib
import joblib
import platform
import sys
import gc
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timezone
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '05_random_forest'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

MODELS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'random_forest'
)

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    MODELS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# Load shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']
FEATURE_BUDGET = config['classical_features']['feature_budget']

MODEL_NAME = 'Random Forest'
REPRESENTATION = 'word_tfidf'
RUN_ID = 'RF_WORD_BASELINE_S42'
SCORE_TYPE = 'positive_class_probability'

DEFAULT_THRESHOLD = float(
    config['evaluation']['default_probability_threshold']
)

random.seed(SEED)
np.random.seed(SEED)

assert len(LABELS) == 6
assert SEED == 42
assert DEFAULT_THRESHOLD == 0.5
assert FEATURE_BUDGET == 50000

assert config['classical_features']['status'] == 'verified'

assert REPRESENTATION in (
    config['classical_features']['representations']
)

assert REGISTRY_PATH.is_file()


# Reuse the shared evaluation module

sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()


def calculate_sha256(path):

    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024), b''
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

feature_manifest_hash = calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)


# Verify all previously registered experiments

registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry_df['run_id'].is_unique

required_previous_runs = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'LR_CHAR_BASELINE_S42',
    'LR_COMBINED_BASELINE_S42',
    'LR_COMBINED_BALANCED_S42',
    'SVM_WORD_BASELINE_S42',
    'SVM_CHAR_BASELINE_S42',
    'SVM_COMBINED_BASELINE_S42',
    'SVM_COMBINED_BALANCED_S42',
    'DT_WORD_BASELINE_S42'
]

assert set(required_previous_runs).issubset(
    set(registry_df['run_id'])
)

assert RUN_ID not in set(registry_df['run_id']), (
    "Random Forest run already registered; "
    "review existing artifacts before starting another run."
)

previous_run_configs = {}
consistency_records = []

for run_id in required_previous_runs:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    assert record['status'] == 'Completed'

    config_path = PROJECT_ROOT / record['config_path']
    model_path = PROJECT_ROOT / record['model_path']

    assert config_path.is_file()
    assert model_path.is_file()

    with open(config_path, 'r') as file:
        run_config = json.load(file)

    assert run_config['labels'] == LABELS
    assert run_config['seed'] == SEED

    assert (
        run_config['evaluation_module_sha256']
        == evaluation_hash
    )

    assert (
        run_config['feature_manifest_sha256']
        == feature_manifest_hash
    )

    assert not run_config['official_test_used']

    previous_run_configs[run_id] = run_config

    consistency_records.append({
        'Run ID': run_id,
        'Label Order Verified': True,
        'Seed Verified': True,
        'Evaluation Module Verified': True,
        'Feature Manifest Verified': True,
        'Model Available': True,
        'Official Test Used': False,
        'Status': 'Verified'
    })

previous_run_verification_df = pd.DataFrame(
    consistency_records
)

print("Previous experiment consistency verification")

display(previous_run_verification_df)

previous_run_verification_df.to_csv(
    TABLES_DIR / 'previous_run_consistency.csv',
    index=False
)


# Verify common dataset fingerprints

training_hashes = {
    run_config['training_dataset_sha256']
    for run_config in previous_run_configs.values()
}

validation_hashes = {
    run_config['validation_dataset_sha256']
    for run_config in previous_run_configs.values()
}

assert len(training_hashes) == 1
assert len(validation_hashes) == 1

EXPECTED_TRAINING_HASH = next(iter(training_hashes))
EXPECTED_VALIDATION_HASH = next(iter(validation_hashes))


# Verify the completed Decision Tree experiment

dt_record = registry_df.loc[
    registry_df['run_id'] == 'DT_WORD_BASELINE_S42'
].iloc[0]

DT_RUN_DIR = PROJECT_ROOT / dt_record['results_path']

dt_manifest_df = pd.read_csv(
    DT_RUN_DIR / 'tables' / 'artifact_manifest.csv'
)

assert len(dt_manifest_df) == 7

assert dt_manifest_df['Run ID'].eq(
    'DT_WORD_BASELINE_S42'
).all()

for _, artifact in dt_manifest_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()
    assert path.stat().st_size == artifact['Size Bytes']

    assert calculate_sha256(path) == artifact['SHA256']

dt_summary = pd.read_csv(
    DT_RUN_DIR / 'tables' / 'run_summary.csv'
).iloc[0]

assert dt_summary['Status'] == 'Completed'
assert dt_summary['Structural Verification'] == 'Passed'

assert np.isclose(
    dt_summary['Macro AP'],
    float(dt_record['validation_macro_ap']),
    rtol=0,
    atol=1e-6
)

assert np.isclose(
    dt_summary['Optimized Macro F1'],
    float(dt_record['validation_macro_f1']),
    rtol=0,
    atol=1e-6
)

print("\nDecision Tree reference verification")

print("Artifact fingerprints verified:", len(dt_manifest_df))
print("Validation Macro AP:", round(dt_summary['Macro AP'], 6))

print("Optimized Macro F1:",
      round(dt_summary['Optimized Macro F1'], 6))

print("Macro AP generalization gap:",
      round(dt_summary['Macro AP Generalization Gap'], 6))

print("Reference experiment: Verified")


# Verify evaluation support for averaged probability scores

toy_labels = ['example_a', 'example_b']

toy_targets = np.array([
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1],
    [1, 0],
    [0, 1]
], dtype=np.int8)

toy_tree_scores = np.array([
    [
        [0.0, 0.1],
        [0.6, 0.2],
        [0.1, 0.8],
        [0.7, 0.9],
        [0.6, 0.2],
        [0.1, 0.8]
    ],
    [
        [0.1, 0.0],
        [0.4, 0.1],
        [0.2, 0.7],
        [0.5, 0.8],
        [0.4, 0.1],
        [0.2, 0.7]
    ],
    [
        [0.2, 0.2],
        [0.2, 0.3],
        [0.3, 0.6],
        [0.3, 0.7],
        [0.2, 0.3],
        [0.3, 0.6]
    ]
], dtype=np.float64)

toy_scores = toy_tree_scores.mean(axis=0)

assert toy_scores.shape == toy_targets.shape
assert np.isfinite(toy_scores).all()
assert (toy_scores >= 0).all()
assert (toy_scores <= 1).all()

toy_default_metrics, toy_default_per_label = (
    evaluate_multilabel(
        toy_targets,
        toy_scores,
        toy_labels,
        thresholds=DEFAULT_THRESHOLD
    )
)

toy_threshold_df = optimize_thresholds(
    toy_targets,
    toy_scores,
    toy_labels
)

toy_thresholds = toy_threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

toy_optimized_metrics, toy_optimized_per_label = (
    evaluate_multilabel(
        toy_targets,
        toy_scores,
        toy_labels,
        thresholds=toy_thresholds
    )
)

assert np.allclose(
    toy_thresholds,
    [0.4, 0.7]
)

assert np.isclose(
    toy_optimized_metrics['Macro F1'],
    1.0
)

assert np.array_equal(
    apply_thresholds(toy_scores, toy_thresholds),
    toy_targets
)

for metric in [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]:

    assert np.isclose(
        toy_default_metrics[metric],
        toy_optimized_metrics[metric],
        rtol=0,
        atol=1e-12
    )

probability_verification_df = pd.DataFrame({
    'Label': toy_labels,
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimal Threshold': toy_thresholds,
    'Default F1':
        toy_default_per_label['F1'].to_numpy(),
    'Optimized F1':
        toy_optimized_per_label['F1'].to_numpy(),
    'Averaged Probability Scores Supported': True,
    'Verification': 'Passed'
})

print("\nEnsemble probability evaluation verification")

display(probability_verification_df.round(6))

probability_verification_df.to_csv(
    TABLES_DIR / 'ensemble_probability_evaluation_verification.csv',
    index=False
)


# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

MODEL_COLORS = {
    'Multinomial Naive Bayes': '#94A3B8',
    'Logistic Regression': '#4C78A8',
    'Linear SVM': '#54A24B',
    'Decision Tree': '#F58518',
    'Random Forest': '#B279A2'
}


# Record environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

print("\nCurrent environment versions")

display(environment_df)


# Verify environment compatibility

feature_environment_path = (
    CLASSICAL_DIR / '00_feature_extraction'
    / 'results' / 'tables' / 'environment_versions.csv'
)

assert feature_environment_path.is_file()

feature_environment_df = pd.read_csv(
    feature_environment_path
)

environment_comparison_df = environment_df.merge(
    feature_environment_df,
    on='Package',
    suffixes=('_Current', '_Feature_Extraction'),
    validate='one_to_one'
)

assert len(environment_comparison_df) == len(environment_df)

environment_comparison_df['Version Match'] = (
    environment_comparison_df['Version_Current']
    == environment_comparison_df['Version_Feature_Extraction']
)

print("\nEnvironment compatibility")

display(environment_comparison_df)

environment_comparison_df.to_csv(
    TABLES_DIR / 'environment_compatibility.csv',
    index=False
)

assert environment_comparison_df['Version Match'].all(), (
    "Environment version mismatch requires review"
)


# Record available CPU and memory resources

import os
import psutil

runtime_resources = {
    'logical_cpu_count': os.cpu_count(),
    'available_cpu_count': (
        len(os.sched_getaffinity(0))
        if hasattr(os, 'sched_getaffinity')
        else os.cpu_count()
    ),
    'total_ram_gib':
        psutil.virtual_memory().total / (1024 ** 3),
    'available_ram_gib':
        psutil.virtual_memory().available / (1024 ** 3),
    'current_process_ram_mib':
        psutil.Process().memory_info().rss / (1024 ** 2)
}

with open(
    LOGS_DIR / 'runtime_resources.json',
    'w'
) as file:

    json.dump(
        runtime_resources,
        file,
        indent=4
    )

print("\nAvailable runtime resources")

display(
    pd.DataFrame([
        runtime_resources
    ]).T.rename(columns={0: 'Value'}).round(4)
)


# Summarize project initialization

print("\nRandom Forest initialization summary")

print("Notebook:", NOTEBOOK_DIR.name)
print("Model:", MODEL_NAME)
print("Planned run:", RUN_ID)
print("Feature representation:", REPRESENTATION)
print("Feature budget:", FEATURE_BUDGET)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("Score type:", SCORE_TYPE)
print("Default threshold:", DEFAULT_THRESHOLD)
print("Results directory:", RESULTS_DIR)
print("Model directory:", MODELS_DIR)

print("Previous experiments verified:",
      len(previous_run_verification_df))

print("Decision Tree fingerprints verified:",
      len(dt_manifest_df))

print("Matching environment packages:",
      environment_comparison_df['Version Match'].sum(),
      "/", len(environment_df))

print("Shared dataset fingerprints: Consistent")
print("Shared evaluation module: Verified")
print("Averaged probability scores: Supported")
print("Feature artifact manifest: Verified")
print("Runtime resources recorded: Yes")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nRandom Forest environment initialized successfully.")

In [ ]:
# Load the shared dataset and feature manifests

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

datasets = {}
dataset_records = {}
verification_records = []


# Verify training and validation datasets

for partition, source_partition, expected_hash in [
    ('train', 'train_representative', EXPECTED_TRAINING_HASH),
    ('validation', 'validation', EXPECTED_VALIDATION_HASH)
]:

    matching_records = dataset_manifest.loc[
        dataset_manifest['Partition'] == source_partition
    ]

    assert len(matching_records) == 1

    record = matching_records.iloc[0]

    dataset_path = PROJECT_ROOT / record['Dataset Path']

    assert dataset_path.is_file()

    dataset_hash = calculate_sha256(dataset_path)

    assert dataset_hash == record['SHA256']
    assert dataset_hash == expected_hash

    df = pd.read_parquet(dataset_path)

    assert len(df) == record['Rows']
    assert df['id'].is_unique
    assert df['group_id'].notna().all()
    assert df['comment_text'].notna().all()
    assert df[LABELS].isin([0, 1]).all().all()


    # Verify feature row alignment

    row_records = feature_manifest.loc[
        (feature_manifest['Artifact Type'] == 'Row Manifest')
        & (feature_manifest['Partition'] == partition)
    ]

    assert len(row_records) == 1

    row_record = row_records.iloc[0]

    row_path = PROJECT_ROOT / row_record['Path']

    assert row_path.is_file()

    assert calculate_sha256(row_path) == row_record['SHA256']

    assert (
        row_record['Source Dataset SHA256']
        == dataset_hash
    )

    row_manifest = pd.read_csv(
        row_path,
        dtype={'id': str}
    )

    assert len(row_manifest) == len(df)

    assert (
        row_manifest['id'].tolist()
        == df['id'].tolist()
    )

    assert (
        row_manifest['group_id'].tolist()
        == df['group_id'].tolist()
    )

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    datasets[partition] = df
    dataset_records[partition] = record

    verification_records.append({
        'Partition': partition,
        'Comments': len(df),
        'Unique IDs': df['id'].nunique(),
        'Dataset Hash Verified': True,
        'Previous Experiments Match': True,
        'Row Manifest Verified': True,
        'Row Alignment Verified': True,
        'Valid Labels': True
    })


# Construct multi-label targets

train_df = datasets['train']
val_df = datasets['validation']

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_val = val_df[LABELS].to_numpy(dtype=np.int8)

assert y_train.shape == (59997, len(LABELS))
assert y_val.shape == (31911, len(LABELS))

assert set(train_df['id']).isdisjoint(val_df['id'])

assert set(train_df['group_id']).isdisjoint(
    val_df['group_id']
)

assert (y_train.sum(axis=0) > 0).all()
assert (y_val.sum(axis=0) > 0).all()

assert (y_train.sum(axis=0) < len(y_train)).all()
assert (y_val.sum(axis=0) < len(y_val)).all()

dataset_verification_df = pd.DataFrame(
    verification_records
)

print("Dataset integrity verification")

display(dataset_verification_df)

dataset_verification_df.to_csv(
    TABLES_DIR / 'dataset_verification.csv',
    index=False
)


# Load and verify the common sparse word features

def load_verified_word_features():

    matrices = {}
    records = []

    for partition in ['train', 'validation']:

        matching_records = feature_manifest.loc[
            (feature_manifest['Artifact Type'] == 'Sparse Matrix')
            & (feature_manifest['Representation'] == REPRESENTATION)
            & (feature_manifest['Partition'] == partition)
        ]

        assert len(matching_records) == 1

        record = matching_records.iloc[0]

        path = PROJECT_ROOT / record['Path']

        assert path.is_file()
        assert calculate_sha256(path) == record['SHA256']

        assert (
            record['Source Dataset SHA256']
            == dataset_records[partition]['SHA256']
        )

        matrix = sp.load_npz(path)

        assert matrix.format == 'csr'
        assert matrix.dtype == np.float32

        assert matrix.shape == (
            len(datasets[partition]),
            FEATURE_BUDGET
        )

        assert np.isfinite(matrix.data).all()
        assert (matrix.data >= 0).all()

        memory_bytes = (
            matrix.data.nbytes
            + matrix.indices.nbytes
            + matrix.indptr.nbytes
        )

        matrices[partition] = matrix

        records.append({
            'Representation': REPRESENTATION,
            'Partition': partition,
            'Comments': matrix.shape[0],
            'Features': matrix.shape[1],
            'Nonzero Elements': matrix.nnz,
            'Data Type': str(matrix.dtype),
            'Sparse Memory (MiB)':
                memory_bytes / (1024 ** 2),
            'Zero Feature Comments':
                int(np.sum(np.diff(matrix.indptr) == 0)),
            'Feature Hash Verified': True,
            'Source Dataset Verified': True,
            'Verification': 'Passed'
        })

    return (
        matrices['train'],
        matrices['validation'],
        pd.DataFrame(records)
    )


loading_start = time.perf_counter()

X_train, X_val, feature_verification_df = (
    load_verified_word_features()
)

feature_loading_time = (
    time.perf_counter() - loading_start
)

print("\nWord TF-IDF feature verification")

display(feature_verification_df.round(4))

feature_verification_df.to_csv(
    TABLES_DIR / 'feature_representation_verification.csv',
    index=False
)


# Verify feature consistency across previous classifiers

comparison_columns = [
    'Representation',
    'Partition',
    'Comments',
    'Features',
    'Nonzero Elements',
    'Data Type',
    'Sparse Memory (MiB)',
    'Zero Feature Comments'
]

consistency_checks = []

for notebook in [
    '02_logistic_regression',
    '03_linear_svm',
    '04_decision_tree'
]:

    previous_feature_df = pd.read_csv(
        CLASSICAL_DIR / notebook
        / 'results' / 'tables'
        / 'feature_representation_verification.csv'
    )

    previous_word_df = previous_feature_df.loc[
        previous_feature_df['Representation']
        == REPRESENTATION
    ]

    current_comparison = (
        feature_verification_df[comparison_columns]
        .sort_values('Partition')
        .reset_index(drop=True)
    )

    previous_comparison = (
        previous_word_df[comparison_columns]
        .sort_values('Partition')
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        current_comparison,
        previous_comparison,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )

    consistency_checks.append({
        'Previous Notebook': notebook,
        'Representation': REPRESENTATION,
        'Training Features Match': True,
        'Validation Features Match': True,
        'Verification': 'Passed'
    })

feature_consistency_df = pd.DataFrame(
    consistency_checks
)

print("\nPrevious experiment feature consistency")

display(feature_consistency_df)

feature_consistency_df.to_csv(
    TABLES_DIR / 'previous_feature_consistency.csv',
    index=False
)


# Record the actual class distribution

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives':
        len(y_train) - y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Validation Negatives':
        len(y_val) - y_val.sum(axis=0)
})

label_support_df['Training Imbalance Ratio'] = (
    label_support_df['Training Negatives']
    / label_support_df['Training Positives']
)

label_support_df['Training Prevalence (%)'] = (
    100 * label_support_df['Training Positives']
    / len(y_train)
)

label_support_df['Validation Prevalence (%)'] = (
    100 * label_support_df['Validation Positives']
    / len(y_val)
)

print("\nLabel support and class imbalance")

display(label_support_df.round(6))

label_support_df.to_csv(
    TABLES_DIR / 'label_support_and_imbalance.csv',
    index=False
)


# Verify identical label support with Decision Tree

dt_label_support_df = pd.read_csv(
    CLASSICAL_DIR / '04_decision_tree'
    / 'results' / 'tables'
    / 'label_support_and_imbalance.csv'
)

pd.testing.assert_frame_equal(
    label_support_df,
    dt_label_support_df,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

print("\nDecision Tree label support consistency: Passed")


# Estimate expected positive samples in full-size bootstrap draws

bootstrap_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Prevalence':
        y_train.mean(axis=0)
})

bootstrap_sample_size = len(y_train)

bootstrap_support_df['Bootstrap Draws'] = (
    bootstrap_sample_size
)

bootstrap_support_df['Expected Positive Draws'] = (
    bootstrap_sample_size
    * bootstrap_support_df['Training Prevalence']
)

bootstrap_support_df['Expected Unique Positive Samples'] = (
    bootstrap_support_df['Training Positives']
    * (
        -np.expm1(
            bootstrap_sample_size
            * np.log1p(-1 / len(y_train))
        )
    )
)

bootstrap_support_df['Probability of No Positive Draws'] = (
    np.exp(
        bootstrap_sample_size
        * np.log1p(
            -bootstrap_support_df['Training Prevalence']
        )
    )
)

print("\nTheoretical full-size bootstrap support")

display(bootstrap_support_df.round(6))

bootstrap_support_df.to_csv(
    TABLES_DIR / 'rf_bootstrap_support_estimates.csv',
    index=False
)


# Record memory usage after loading the datasets

training_memory_mib = (
    X_train.data.nbytes
    + X_train.indices.nbytes
    + X_train.indptr.nbytes
) / (1024 ** 2)

validation_memory_mib = (
    X_val.data.nbytes
    + X_val.indices.nbytes
    + X_val.indptr.nbytes
) / (1024 ** 2)

gc.collect()

loaded_runtime_resources = {
    'available_cpu_count':
        runtime_resources['available_cpu_count'],
    'available_ram_gib':
        psutil.virtual_memory().available / (1024 ** 3),
    'current_process_ram_mib':
        psutil.Process().memory_info().rss / (1024 ** 2),
    'training_sparse_memory_mib':
        float(training_memory_mib),
    'validation_sparse_memory_mib':
        float(validation_memory_mib),
    'total_sparse_memory_mib':
        float(training_memory_mib + validation_memory_mib),
    'feature_loading_time_seconds':
        float(feature_loading_time)
}

with open(
    LOGS_DIR / 'loaded_runtime_resources.json',
    'w'
) as file:

    json.dump(
        loaded_runtime_resources,
        file,
        indent=4
    )

print("\nRuntime resources after loading data")

display(
    pd.DataFrame([
        loaded_runtime_resources
    ]).T.rename(columns={0: 'Value'}).round(4)
)


# Summarize dataset and feature verification

print("\nRandom Forest dataset verification summary")

print("Training label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)

print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_val.shape)

print("Training nonzero elements:", X_train.nnz)
print("Validation nonzero elements:", X_val.nnz)

print("Total sparse matrix memory (MiB):",
      round(training_memory_mib + validation_memory_mib, 4))

print("Feature loading time (s):",
      round(feature_loading_time, 4))

print("Available RAM after loading (GiB):",
      round(
          loaded_runtime_resources['available_ram_gib'],
          4
      ))

print("Sparse matrices verified:",
      len(feature_verification_df))

print("Previous feature consistency checks:",
      len(feature_consistency_df))

print("Decision Tree label support: Verified")
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Bootstrap support estimates: Saved")
print("Dense matrix conversion: No")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nRandom Forest dataset and feature verification completed successfully.")

In [ ]:
# Define the resource-controlled Random Forest configuration

RF_PARAMS = {
    'n_estimators': 50,
    'criterion': 'gini',
    'max_depth': 30,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'min_weight_fraction_leaf': 0.0,
    'max_features': 'sqrt',
    'max_leaf_nodes': 512,
    'min_impurity_decrease': 0.0,
    'bootstrap': True,
    'oob_score': False,
    'n_jobs': min(2, runtime_resources['available_cpu_count']),
    'random_state': SEED,
    'verbose': 0,
    'warm_start': False,
    'class_weight': None,
    'ccp_alpha': 0.0,
    'max_samples': None
}

OVR_PARAMS = {
    'n_jobs': 1
}

experiment_spec = {
    'run_id': RUN_ID,
    'representation': REPRESENTATION,
    'class_weight': None,
    'purpose': 'Resource-controlled nonlinear ensemble baseline'
}


# Verify resource availability and predefined settings

assert RF_PARAMS['n_jobs'] >= 1
assert RF_PARAMS['n_jobs'] <= runtime_resources['available_cpu_count']

assert RF_PARAMS['bootstrap']
assert RF_PARAMS['max_samples'] is None
assert RF_PARAMS['class_weight'] is None
assert RF_PARAMS['n_estimators'] == 50

assert OVR_PARAMS['n_jobs'] == 1


# Verify comparable structural restrictions with Decision Tree

dt_config = previous_run_configs['DT_WORD_BASELINE_S42']

dt_params = dt_config['model_parameters']

matched_parameters = [
    'criterion',
    'max_depth',
    'min_samples_split',
    'min_samples_leaf',
    'min_weight_fraction_leaf',
    'max_leaf_nodes',
    'min_impurity_decrease',
    'class_weight',
    'ccp_alpha',
    'random_state'
]

parameter_comparison_records = []

for parameter in matched_parameters:

    assert RF_PARAMS[parameter] == dt_params[parameter]

    parameter_comparison_records.append({
        'Parameter': parameter,
        'Decision Tree': str(dt_params[parameter]),
        'Random Forest': str(RF_PARAMS[parameter]),
        'Comparison': 'Matched'
    })

parameter_comparison_records.extend([
    {
        'Parameter': 'max_features',
        'Decision Tree': str(dt_params['max_features']),
        'Random Forest': str(RF_PARAMS['max_features']),
        'Comparison': 'Algorithm-specific difference'
    },
    {
        'Parameter': 'Trees per label',
        'Decision Tree': '1',
        'Random Forest': str(RF_PARAMS['n_estimators']),
        'Comparison': 'Algorithm-specific difference'
    },
    {
        'Parameter': 'Bootstrap sampling',
        'Decision Tree': 'False',
        'Random Forest': str(RF_PARAMS['bootstrap']),
        'Comparison': 'Algorithm-specific difference'
    }
])

dt_rf_parameter_comparison_df = pd.DataFrame(
    parameter_comparison_records
)

print("Decision Tree and Random Forest configuration comparison")

display(dt_rf_parameter_comparison_df)

dt_rf_parameter_comparison_df.to_csv(
    TABLES_DIR / 'dt_rf_parameter_comparison.csv',
    index=False
)


# Construct the One-vs-Rest Random Forest model

def build_rf_model():

    estimator = RandomForestClassifier(
        **RF_PARAMS
    )

    return OneVsRestClassifier(
        estimator,
        **OVR_PARAMS
    )


rf_model = build_rf_model()

assert isinstance(
    rf_model.estimator,
    RandomForestClassifier
)

for parameter, value in RF_PARAMS.items():

    assert (
        rf_model.estimator.get_params()[parameter]
        == value
    )

assert rf_model.n_jobs == 1
assert hasattr(rf_model, 'predict_proba')


# Record hyperparameter rationale before training

hyperparameter_rationale = [
    (
        'n_estimators',
        '50',
        'Use a moderate ensemble size to restrict CPU training '
        'cost while allowing probability averaging across trees.'
    ),
    (
        'criterion',
        'gini',
        'Maintain the impurity criterion used in Decision Tree.'
    ),
    (
        'max_depth',
        '30',
        'Match Decision Tree depth restrictions and prevent '
        'unrestricted tree growth.'
    ),
    (
        'min_samples_split',
        '10',
        'Restrict splitting of small internal nodes.'
    ),
    (
        'min_samples_leaf',
        '5',
        'Avoid extremely small terminal leaves and preserve '
        'the Decision Tree structural restriction.'
    ),
    (
        'max_leaf_nodes',
        '512',
        'Bound the maximum structural complexity of each tree.'
    ),
    (
        'max_features',
        'sqrt',
        'Introduce random feature selection to diversify trees '
        'and reduce the usual number of split candidates.'
    ),
    (
        'bootstrap',
        'True',
        'Train trees on independently generated bootstrap '
        'samples as part of standard Random Forest learning.'
    ),
    (
        'max_samples',
        'None',
        'Retain full-size bootstrap draws to avoid further '
        'reducing rare-label support.'
    ),
    (
        'class_weight',
        'None',
        'Maintain the common unweighted classical baseline.'
    ),
    (
        'oob_score',
        'False',
        'Use the existing independent validation partition '
        'rather than adding an out-of-bag evaluation protocol.'
    ),
    (
        'ccp_alpha',
        '0.0',
        'Avoid additional pruning beyond predefined '
        'structural restrictions.'
    ),
    (
        'random_state',
        str(SEED),
        'Control estimator and bootstrap randomness.'
    ),
    (
        'n_jobs',
        str(RF_PARAMS['n_jobs']),
        'Use available CPU cores for parallel tree fitting.'
    ),
    (
        'OvR n_jobs',
        '1',
        'Train label-specific forests sequentially to avoid '
        'nested parallelization and excessive memory usage.'
    ),
    (
        'warm_start',
        'False',
        'Train a fresh independent ensemble rather than '
        'reusing trees from earlier experiments.'
    )
]

rf_hyperparameter_rationale_df = pd.DataFrame(
    hyperparameter_rationale,
    columns=['Parameter', 'Value', 'Justification']
)

print("\nRandom Forest hyperparameter rationale")

display(rf_hyperparameter_rationale_df)

rf_hyperparameter_rationale_df.to_csv(
    TABLES_DIR / 'rf_hyperparameter_rationale.csv',
    index=False
)


# Calculate theoretical structural limits

trees_per_label = RF_PARAMS['n_estimators']

total_trees = trees_per_label * len(LABELS)

maximum_leaves = RF_PARAMS['max_leaf_nodes']

maximum_nodes = 2 * maximum_leaves - 1

candidate_features_per_split = max(
    1,
    int(np.sqrt(FEATURE_BUDGET))
)

maximum_total_nodes = maximum_nodes * total_trees

rf_structural_limits_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Training Negatives': len(y_train) - y_train.sum(axis=0),
    'Trees per Label': trees_per_label,
    'Maximum Depth per Tree': RF_PARAMS['max_depth'],
    'Maximum Leaves per Tree': maximum_leaves,
    'Maximum Nodes per Tree': maximum_nodes,
    'Maximum Nodes per Forest': maximum_nodes * trees_per_label,
    'Minimum Samples per Leaf': RF_PARAMS['min_samples_leaf'],
    'Nominal Candidate Features per Split':
        candidate_features_per_split
})

print("\nPer-label Random Forest structural limits")

display(rf_structural_limits_df)

rf_structural_limits_df.to_csv(
    TABLES_DIR / 'rf_structural_limits.csv',
    index=False
)


# Verify expected bootstrap support for rare labels

assert bootstrap_support_df['Bootstrap Draws'].eq(
    len(y_train)
).all()

rare_labels = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

rare_bootstrap_support_df = bootstrap_support_df.loc[
    bootstrap_support_df['Label'].isin(rare_labels)
].copy()

assert len(rare_bootstrap_support_df) == 3

print("\nExpected rare-label support per tree")

display(
    rare_bootstrap_support_df[[
        'Label',
        'Training Positives',
        'Expected Positive Draws',
        'Expected Unique Positive Samples',
        'Probability of No Positive Draws'
    ]]
)

rare_bootstrap_support_df.to_csv(
    TABLES_DIR / 'rf_rare_label_bootstrap_support.csv',
    index=False
)


# Save the predefined experiment configuration

rf_experiment_config = {
    'notebook': NOTEBOOK_DIR.name,
    'run_id': RUN_ID,
    'model': MODEL_NAME,
    'estimator': 'RandomForestClassifier',
    'classification_strategy': 'One-vs-Rest',
    'labels': LABELS,
    'num_labels': len(LABELS),
    'seed': SEED,
    'sampling_strategy': 'representative_multilabel',
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'feature_representation': REPRESENTATION,
    'feature_budget': FEATURE_BUDGET,
    'model_parameters': RF_PARAMS,
    'wrapper_parameters': OVR_PARAMS,
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'threshold_comparison_operator': '>=',
    'threshold_optimization_metric': 'per_label_f1',
    'threshold_tie_breaking': 'highest_threshold',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'probability_calibration': False,
    'hyperparameter_selection_strategy':
        'predefined_resource_controlled_baseline',
    'hyperparameter_search_performed': False,
    'feature_representation_search_performed': False,
    'class_weighting_experiment_planned': False,
    'trees_per_label': trees_per_label,
    'total_trees': total_trees,
    'maximum_nodes_per_tree': maximum_nodes,
    'maximum_total_nodes': maximum_total_nodes,
    'nominal_candidate_features_per_split':
        candidate_features_per_split,
    'bootstrap_draws_per_tree': len(y_train),
    'reference_decision_tree_run':
        'DT_WORD_BASELINE_S42',
    'matched_tree_parameters': matched_parameters,
    'runtime_resources': loaded_runtime_resources,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'feature_manifest_sha256': feature_manifest_hash,
    'evaluation_module_sha256': evaluation_hash,
    'sklearn_version': sklearn.__version__,
    'official_test_used': False
}

RF_EXPERIMENT_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'rf_experiment_config.json'
)

with open(
    RF_EXPERIMENT_CONFIG_PATH, 'w'
) as file:

    json.dump(
        rf_experiment_config,
        file,
        indent=4
    )

with open(
    RF_EXPERIMENT_CONFIG_PATH, 'r'
) as file:

    saved_rf_config = json.load(file)

assert saved_rf_config == rf_experiment_config


# Record the planned experiment

rf_experiment_plan_df = pd.DataFrame([{
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Representation': REPRESENTATION,
    'Class Weight': 'None',
    'Training Samples': len(y_train),
    'Validation Samples': len(y_val),
    'Features': FEATURE_BUDGET,
    'Binary Forests': len(LABELS),
    'Trees per Forest': trees_per_label,
    'Total Trees': total_trees,
    'Criterion': RF_PARAMS['criterion'],
    'Maximum Depth': RF_PARAMS['max_depth'],
    'Maximum Leaves per Tree': maximum_leaves,
    'Minimum Samples per Leaf': RF_PARAMS['min_samples_leaf'],
    'Nominal Candidate Features per Split':
        candidate_features_per_split,
    'Bootstrap': RF_PARAMS['bootstrap'],
    'Bootstrap Draws per Tree': len(y_train),
    'Forest Parallel Jobs': RF_PARAMS['n_jobs'],
    'OvR Parallel Jobs': OVR_PARAMS['n_jobs'],
    'Default Threshold': DEFAULT_THRESHOLD,
    'Primary Selection Metric': 'Macro AP',
    'Purpose': experiment_spec['purpose'],
    'Status': 'Planned'
}])

print("\nRandom Forest experiment plan")

display(rf_experiment_plan_df.T)

rf_experiment_plan_df.to_csv(
    TABLES_DIR / 'rf_experiment_plan.csv',
    index=False
)


# Initialize result storage

rf_results = {}


# Summarize the predefined experiment

print("\nRandom Forest configuration summary")

print("Planned run:", RUN_ID)
print("Estimator:", type(rf_model.estimator).__name__)
print("Classification strategy: One-vs-Rest")
print("Binary forests:", len(LABELS))
print("Trees per forest:", trees_per_label)
print("Total trees planned:", total_trees)

print("Feature representation:", REPRESENTATION)
print("Training samples:", len(y_train))
print("Validation samples:", len(y_val))
print("Feature count:", FEATURE_BUDGET)

print("Criterion:", RF_PARAMS['criterion'])
print("Maximum depth per tree:", RF_PARAMS['max_depth'])
print("Maximum leaves per tree:", maximum_leaves)
print("Maximum nodes per tree:", maximum_nodes)
print("Maximum total nodes:", maximum_total_nodes)

print("Minimum samples per leaf:",
      RF_PARAMS['min_samples_leaf'])

print("Nominal candidate features per split:",
      candidate_features_per_split)

print("Bootstrap sampling:", RF_PARAMS['bootstrap'])
print("Bootstrap draws per tree:", len(y_train))
print("Class weighting:", RF_PARAMS['class_weight'])
print("Random Forest parallel jobs:", RF_PARAMS['n_jobs'])
print("One-vs-Rest parallel jobs:", OVR_PARAMS['n_jobs'])

print("Default threshold:", DEFAULT_THRESHOLD)
print("Primary selection metric: Macro AP")

print("Decision Tree matched parameters:",
      len(matched_parameters))

print("Hyperparameter search: Not performed")
print("Configuration saved and verified: Yes")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nRandom Forest configuration completed successfully.")

In [ ]:
# Prepare experiment-specific artifact directories

import threading

RUN_DIR = RESULTS_DIR / RUN_ID

RUN_TABLES_DIR = RUN_DIR / 'tables'
RUN_PREDICTIONS_DIR = RUN_DIR / 'predictions'
RUN_LOGS_DIR = RUN_DIR / 'logs'

MODEL_PATH = MODELS_DIR / f'{RUN_ID}.joblib'

assert not MODEL_PATH.exists(), (
    "Model already exists. Review saved artifacts before retraining."
)

for folder in [
    RUN_TABLES_DIR,
    RUN_PREDICTIONS_DIR,
    RUN_LOGS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

assert X_train.shape == (59997, FEATURE_BUDGET)
assert X_val.shape == (31911, FEATURE_BUDGET)

assert sp.issparse(X_train)
assert sp.issparse(X_val)

assert X_train.dtype == np.float32
assert X_val.dtype == np.float32

assert y_train.shape == (len(train_df), len(LABELS))
assert y_val.shape == (len(val_df), len(LABELS))


# Verify the saved experimental configuration

with open(RF_EXPERIMENT_CONFIG_PATH, 'r') as file:
    saved_rf_config = json.load(file)

assert saved_rf_config['model_parameters'] == RF_PARAMS
assert saved_rf_config['wrapper_parameters'] == OVR_PARAMS
assert not saved_rf_config['official_test_used']

rf_model = build_rf_model()

print("Starting experiment:", RUN_ID)
print("Representation:", REPRESENTATION)
print("Class weight:", RF_PARAMS['class_weight'])
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Binary forests:", len(LABELS))
print("Trees per forest:", trees_per_label)
print("Total trees:", total_trees)
print("Parallel CPU jobs:", RF_PARAMS['n_jobs'])


# Monitor sampled process memory during training

process = psutil.Process()

memory_before_training_mib = (
    process.memory_info().rss / (1024 ** 2)
)

memory_samples = [memory_before_training_mib]

monitor_stop = threading.Event()


def monitor_memory():

    while not monitor_stop.wait(0.5):

        memory_samples.append(
            process.memory_info().rss / (1024 ** 2)
        )


memory_monitor = threading.Thread(
    target=monitor_memory,
    daemon=True
)

memory_monitor.start()

training_start_time = datetime.now(
    timezone.utc
).isoformat()

training_start = time.perf_counter()


# Convert training features to CSC and fit the model

try:

    conversion_start = time.perf_counter()

    X_train_csc = X_train.tocsc()
    X_train_csc.sort_indices()

    feature_conversion_time = (
        time.perf_counter() - conversion_start
    )

    assert X_train_csc.shape == X_train.shape
    assert X_train_csc.nnz == X_train.nnz
    assert X_train_csc.dtype == np.float32

    print("\nSparse training format:", X_train_csc.format)

    print("Sparse conversion time (s):",
          round(feature_conversion_time, 6))

    print("\nTraining Random Forest classifiers...")

    fit_start = time.perf_counter()

    with warnings.catch_warnings(record=True) as captured_warnings:

        warnings.simplefilter('always')

        rf_model.fit(
            X_train_csc,
            y_train
        )

    model_fit_time = (
        time.perf_counter() - fit_start
    )

    training_time = (
        time.perf_counter() - training_start
    )

finally:

    monitor_stop.set()
    memory_monitor.join()


training_end_time = datetime.now(
    timezone.utc
).isoformat()

memory_after_fit_mib = (
    process.memory_info().rss / (1024 ** 2)
)

memory_samples.append(memory_after_fit_mib)

sampled_peak_memory_mib = max(memory_samples)

print("\nModel training completed.")

print("Model fitting time (s):",
      round(model_fit_time, 6))

print("Total training time including CSC conversion (s):",
      round(training_time, 6))

del X_train_csc

gc.collect()


# Record training warnings

training_warning_records = [
    {
        'Category': warning.category.__name__,
        'Message': str(warning.message)
    }
    for warning in captured_warnings
]

with open(
    RUN_LOGS_DIR / 'training_warnings.json',
    'w'
) as file:

    json.dump(
        training_warning_records,
        file,
        indent=4
    )

print("\nTraining warnings:",
      len(training_warning_records))

if training_warning_records:

    display(
        pd.DataFrame(training_warning_records)
        .value_counts()
        .reset_index(name='Count')
    )


# Verify One-vs-Rest classifier alignment

assert rf_model.multilabel_
assert len(rf_model.estimators_) == len(LABELS)

assert np.array_equal(
    rf_model.classes_,
    np.arange(len(LABELS))
)

assert rf_model.n_features_in_ == FEATURE_BUDGET


# Verify every fitted tree and its bootstrap support

tree_records = []
forest_records = []

for label_index, (label, forest) in enumerate(
    zip(LABELS, rf_model.estimators_)
):

    assert isinstance(
        forest,
        RandomForestClassifier
    )

    assert np.array_equal(
        forest.classes_,
        np.array([0, 1])
    )

    assert forest.n_features_in_ == FEATURE_BUDGET

    assert len(forest.estimators_) == trees_per_label

    for parameter, value in RF_PARAMS.items():

        assert forest.get_params()[parameter] == value

    bootstrap_samples = forest.estimators_samples_

    assert len(bootstrap_samples) == trees_per_label

    forest_tree_records = []
    forest_used_features = set()

    for tree_index, (estimator, sample_indices) in enumerate(
        zip(forest.estimators_, bootstrap_samples)
    ):

        tree = estimator.tree_

        leaf_mask = tree.children_left == -1
        internal_mask = ~leaf_mask

        depth = estimator.get_depth()
        leaf_count = estimator.get_n_leaves()
        node_count = tree.node_count

        leaf_samples = tree.n_node_samples[leaf_mask]

        used_features = tree.feature[internal_mask]

        unique_sample_indices = np.unique(
            sample_indices
        )

        positive_draws = int(
            y_train[sample_indices, label_index].sum()
        )

        unique_positive_samples = int(
            y_train[
                unique_sample_indices,
                label_index
            ].sum()
        )

        assert estimator.n_features_in_ == FEATURE_BUDGET

        assert (
            estimator.max_features_
            == candidate_features_per_split
        )

        assert len(sample_indices) == len(y_train)

        assert (sample_indices >= 0).all()
        assert (sample_indices < len(y_train)).all()

        assert np.isfinite(tree.threshold).all()
        assert np.isfinite(tree.impurity).all()
        assert np.isfinite(tree.value).all()

        assert depth <= RF_PARAMS['max_depth']
        assert leaf_count <= maximum_leaves
        assert node_count <= maximum_nodes

        assert node_count == 2 * leaf_count - 1
        assert leaf_mask.sum() == leaf_count

        assert (
            tree.children_right[leaf_mask] == -1
        ).all()

        assert leaf_samples.min() >= (
            RF_PARAMS['min_samples_leaf']
        )

        # Unweighted counts exclude repeated bootstrap draws

        assert tree.n_node_samples[0] == len(
            unique_sample_indices
        )

        assert leaf_samples.sum() == len(
            unique_sample_indices
        )

        # Weighted counts include bootstrap multiplicities

        assert np.isclose(
            tree.weighted_n_node_samples[0],
            len(sample_indices),
            rtol=0,
            atol=1e-8
        )

        assert np.isclose(
            tree.weighted_n_node_samples[leaf_mask].sum(),
            len(sample_indices),
            rtol=0,
            atol=1e-8
        )

        if internal_mask.any():

            assert (
                tree.n_node_samples[internal_mask]
                >= RF_PARAMS['min_samples_split']
            ).all()

            assert (used_features >= 0).all()
            assert (used_features < FEATURE_BUDGET).all()

        forest_used_features.update(
            used_features.tolist()
        )

        tree_record = {
            'Label': label,
            'Tree Index': tree_index + 1,
            'Tree Random State': int(estimator.random_state),
            'Actual Depth': depth,
            'Leaf Count': leaf_count,
            'Node Count': node_count,
            'Minimum Leaf Samples': int(leaf_samples.min()),
            'Maximum Leaf Samples': int(leaf_samples.max()),
            'Mean Leaf Samples': float(leaf_samples.mean()),
            'Median Leaf Samples': float(np.median(leaf_samples)),
            'Unique Split Features':
                int(len(np.unique(used_features))),
            'Bootstrap Draws': len(sample_indices),
            'Unique Bootstrap Samples':
                len(unique_sample_indices),
            'Positive Bootstrap Draws': positive_draws,
            'Unique Positive Bootstrap Samples':
                unique_positive_samples,
            'Root Impurity': float(tree.impurity[0]),
            'Depth Limit Reached':
                depth == RF_PARAMS['max_depth'],
            'Leaf Limit Reached':
                leaf_count == maximum_leaves,
            'Both Classes Present':
                0 < positive_draws < len(sample_indices),
            'Structural Verification': 'Passed'
        }

        forest_tree_records.append(tree_record)

        tree_records.append(tree_record)

    forest_tree_df = pd.DataFrame(
        forest_tree_records
    )

    forest_records.append({
        'Label': label,
        'Training Positives':
            int(y_train[:, label_index].sum()),
        'Trees': len(forest.estimators_),
        'Mean Depth':
            float(forest_tree_df['Actual Depth'].mean()),
        'Maximum Depth':
            int(forest_tree_df['Actual Depth'].max()),
        'Mean Leaves':
            float(forest_tree_df['Leaf Count'].mean()),
        'Total Leaves':
            int(forest_tree_df['Leaf Count'].sum()),
        'Total Nodes':
            int(forest_tree_df['Node Count'].sum()),
        'Maximum Nodes per Tree':
            int(forest_tree_df['Node Count'].max()),
        'Minimum Leaf Samples':
            int(forest_tree_df['Minimum Leaf Samples'].min()),
        'Unique Features Across Forest':
            len(forest_used_features),
        'Mean Unique Bootstrap Samples':
            float(forest_tree_df['Unique Bootstrap Samples'].mean()),
        'Mean Positive Bootstrap Draws':
            float(forest_tree_df['Positive Bootstrap Draws'].mean()),
        'Mean Unique Positive Bootstrap Samples':
            float(
                forest_tree_df[
                    'Unique Positive Bootstrap Samples'
                ].mean()
            ),
        'Minimum Unique Positive Bootstrap Samples':
            int(
                forest_tree_df[
                    'Unique Positive Bootstrap Samples'
                ].min()
            ),
        'Trees Reaching Depth Limit':
            int(forest_tree_df['Depth Limit Reached'].sum()),
        'Trees Reaching Leaf Limit':
            int(forest_tree_df['Leaf Limit Reached'].sum()),
        'Trees Missing a Class':
            int((~forest_tree_df['Both Classes Present']).sum()),
        'Class Alignment': 'Passed',
        'Structural Verification': 'Passed'
    })

    print(
        f"{label}: "
        f"{len(forest.estimators_)} trees verified"
    )

    del bootstrap_samples


# Save individual tree and forest diagnostics

rf_tree_structure_df = pd.DataFrame(
    tree_records
)

rf_structure_df = pd.DataFrame(
    forest_records
)

assert len(rf_tree_structure_df) == total_trees
assert len(rf_structure_df) == len(LABELS)

assert rf_tree_structure_df[
    'Structural Verification'
].eq('Passed').all()

assert not rf_tree_structure_df.duplicated(
    subset=['Label', 'Tree Index']
).any()

assert rf_structure_df['Trees'].eq(
    trees_per_label
).all()

print("\nPer-label Random Forest structural verification")

display(rf_structure_df.round(6))

rf_tree_structure_df.to_csv(
    RUN_TABLES_DIR / 'individual_tree_structure.csv',
    index=False
)

rf_structure_df.to_csv(
    RUN_TABLES_DIR / 'classifier_structure_summary.csv',
    index=False
)


# Compare observed bootstrap support with theoretical expectations

bootstrap_comparison_df = (
    rf_structure_df[[
        'Label',
        'Mean Positive Bootstrap Draws',
        'Mean Unique Positive Bootstrap Samples',
        'Minimum Unique Positive Bootstrap Samples',
        'Trees Missing a Class'
    ]]
    .merge(
        bootstrap_support_df[[
            'Label',
            'Expected Positive Draws',
            'Expected Unique Positive Samples'
        ]],
        on='Label',
        validate='one_to_one'
    )
)

bootstrap_comparison_df['Unique Positive Support Difference'] = (
    bootstrap_comparison_df[
        'Mean Unique Positive Bootstrap Samples'
    ]
    - bootstrap_comparison_df[
        'Expected Unique Positive Samples'
    ]
)

print("\nObserved versus expected bootstrap support")

display(bootstrap_comparison_df.round(6))

bootstrap_comparison_df.to_csv(
    RUN_TABLES_DIR / 'bootstrap_support_verification.csv',
    index=False
)


# Summarize actual ensemble complexity

rf_complexity_df = pd.DataFrame([
    {
        'Property': 'Binary forests',
        'Actual': len(rf_model.estimators_),
        'Configured Limit': len(LABELS)
    },
    {
        'Property': 'Total individual trees',
        'Actual': len(rf_tree_structure_df),
        'Configured Limit': total_trees
    },
    {
        'Property': 'Maximum observed tree depth',
        'Actual': int(rf_tree_structure_df['Actual Depth'].max()),
        'Configured Limit': RF_PARAMS['max_depth']
    },
    {
        'Property': 'Total leaves',
        'Actual': int(rf_tree_structure_df['Leaf Count'].sum()),
        'Configured Limit': maximum_leaves * total_trees
    },
    {
        'Property': 'Total nodes',
        'Actual': int(rf_tree_structure_df['Node Count'].sum()),
        'Configured Limit': maximum_total_nodes
    },
    {
        'Property': 'Maximum leaves in one tree',
        'Actual': int(rf_tree_structure_df['Leaf Count'].max()),
        'Configured Limit': maximum_leaves
    },
    {
        'Property': 'Maximum nodes in one tree',
        'Actual': int(rf_tree_structure_df['Node Count'].max()),
        'Configured Limit': maximum_nodes
    }
])

print("\nOverall Random Forest complexity")

display(rf_complexity_df)

rf_complexity_df.to_csv(
    RUN_TABLES_DIR / 'model_complexity_summary.csv',
    index=False
)


# Record sampled training memory usage

memory_after_cleanup_mib = (
    process.memory_info().rss / (1024 ** 2)
)

memory_summary = {
    'measurement_type': 'sampled_process_rss',
    'sampling_interval_seconds': 0.5,
    'memory_samples': len(memory_samples),
    'memory_before_training_mib':
        float(memory_before_training_mib),
    'sampled_peak_training_memory_mib':
        float(sampled_peak_memory_mib),
    'memory_after_fit_mib':
        float(memory_after_fit_mib),
    'memory_after_cleanup_mib':
        float(memory_after_cleanup_mib),
    'sampled_peak_increase_mib':
        float(
            sampled_peak_memory_mib
            - memory_before_training_mib
        )
}

with open(
    RUN_LOGS_DIR / 'training_memory_usage.json',
    'w'
) as file:

    json.dump(
        memory_summary,
        file,
        indent=4
    )

print("\nTraining memory usage")

display(
    pd.DataFrame([memory_summary]).T
    .rename(columns={0: 'Value'})
)


# Save the trained model without overwriting an existing model

assert not MODEL_PATH.exists()

temporary_model_path = MODEL_PATH.with_suffix(
    '.joblib.tmp'
)

joblib.dump(
    rf_model,
    temporary_model_path,
    compress=3
)

temporary_model_path.replace(MODEL_PATH)

assert MODEL_PATH.is_file()
assert MODEL_PATH.stat().st_size > 0

model_size_mib = (
    MODEL_PATH.stat().st_size / (1024 ** 2)
)

model_hash = calculate_sha256(MODEL_PATH)


# Save training and structural verification metadata

training_log = {
    'run_id': RUN_ID,
    'model': MODEL_NAME,
    'feature_representation': REPRESENTATION,
    'classification_strategy': 'One-vs-Rest',
    'training_samples': len(y_train),
    'validation_samples': len(y_val),
    'features': FEATURE_BUDGET,
    'binary_classifiers': len(rf_model.estimators_),
    'trees_per_forest': trees_per_label,
    'total_trees': len(rf_tree_structure_df),
    'training_start_time': training_start_time,
    'training_end_time': training_end_time,
    'feature_loading_time_seconds':
        float(feature_loading_time),
    'sparse_conversion_time_seconds':
        float(feature_conversion_time),
    'model_fit_time_seconds':
        float(model_fit_time),
    'training_time_seconds':
        float(training_time),
    'training_warnings':
        len(training_warning_records),
    'maximum_observed_depth':
        int(rf_tree_structure_df['Actual Depth'].max()),
    'total_leaves':
        int(rf_tree_structure_df['Leaf Count'].sum()),
    'total_nodes':
        int(rf_tree_structure_df['Node Count'].sum()),
    'trees_missing_a_class':
        int(rf_structure_df['Trees Missing a Class'].sum()),
    'memory_measurement': memory_summary,
    'all_structural_checks_passed': True,
    'class_alignment_verified': True,
    'bootstrap_support_verified': True,
    'dense_matrix_conversion': False,
    'model_path':
        str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    'model_size_mib':
        float(model_size_mib),
    'model_sha256': model_hash,
    'model_reconstruction_verified': False,
    'validation_evaluation_completed': False,
    'threshold_optimization_completed': False,
    'official_test_used': False,
    'status': 'Trained'
}

TRAINING_LOG_PATH = (
    RUN_LOGS_DIR / 'training_log.json'
)

with open(TRAINING_LOG_PATH, 'w') as file:

    json.dump(
        training_log,
        file,
        indent=4
    )


# Summarize completed Random Forest training

print("\nRandom Forest training summary")

print("Run ID:", RUN_ID)

print("Binary forests trained:",
      len(rf_model.estimators_))

print("Individual trees trained:",
      len(rf_tree_structure_df))

print("Individual structural checks passed:",
      rf_tree_structure_df[
          'Structural Verification'
      ].eq('Passed').sum(),
      "/", total_trees)

print("Maximum observed depth:",
      rf_tree_structure_df['Actual Depth'].max())

print("Total leaves:",
      rf_tree_structure_df['Leaf Count'].sum())

print("Total nodes:",
      rf_tree_structure_df['Node Count'].sum())

print("Trees reaching depth limit:",
      rf_tree_structure_df['Depth Limit Reached'].sum())

print("Trees reaching leaf limit:",
      rf_tree_structure_df['Leaf Limit Reached'].sum())

print("Minimum observed leaf samples:",
      rf_tree_structure_df['Minimum Leaf Samples'].min())

print("Trees missing a class:",
      rf_structure_df['Trees Missing a Class'].sum())

print("Model fitting time (s):",
      round(model_fit_time, 6))

print("Total training time (s):",
      round(training_time, 6))

print("Sampled peak process memory (MiB):",
      round(sampled_peak_memory_mib, 4))

print("Sampled peak memory increase (MiB):",
      round(memory_summary['sampled_peak_increase_mib'], 4))

print("Training warnings:",
      len(training_warning_records))

print("Model size (MiB):",
      round(model_size_mib, 6))

print("Model saved:", MODEL_PATH)
print("Class alignment: Verified")
print("Bootstrap support: Verified")
print("Dense matrix conversion: No")
print("Model reconstruction: Pending")
print("Validation evaluation: Pending")
print("Threshold optimization: Pending")
print("Official test dataset: Not loaded")

print("\nRandom Forest training completed successfully.")

In [ ]:
# Generate reproducible probability scores

assert MODEL_PATH.is_file()
assert calculate_sha256(MODEL_PATH) == model_hash
assert len(rf_model.estimators_) == len(LABELS)


def predict_rf_probabilities(model, features):

    forests = model.estimators_
    original_jobs = [forest.n_jobs for forest in forests]

    try:

        for forest in forests:
            forest.n_jobs = 1

        scores = model.predict_proba(features)

        return np.asarray(scores, dtype=np.float64)

    finally:

        for forest, jobs in zip(forests, original_jobs):
            forest.n_jobs = jobs


prediction_start = time.perf_counter()

val_scores = predict_rf_probabilities(
    rf_model, X_val
)

validation_prediction_time = (
    time.perf_counter() - prediction_start
)

prediction_start = time.perf_counter()

train_scores = predict_rf_probabilities(
    rf_model, X_train
)

training_prediction_time = (
    time.perf_counter() - prediction_start
)

for scores, targets in [
    (train_scores, y_train),
    (val_scores, y_val)
]:

    assert scores.shape == targets.shape
    assert np.isfinite(scores).all()
    assert (scores >= 0).all()
    assert (scores <= 1).all()


# Verify individual forest probability averaging

averaging_errors = []

for i, forest in enumerate(rf_model.estimators_):

    assert np.array_equal(
        forest.classes_,
        np.array([0, 1])
    )

    tree_scores = np.column_stack([
        estimator.predict_proba(X_val[:100])[:, 1]
        for estimator in forest.estimators_
    ])

    averaged_scores = tree_scores.mean(axis=1)

    np.testing.assert_allclose(
        val_scores[:100, i],
        averaged_scores,
        rtol=0,
        atol=1e-12
    )

    averaging_errors.append(
        float(
            np.abs(
                val_scores[:100, i] - averaged_scores
            ).max()
        )
    )

averaging_verification_df = pd.DataFrame({
    'Label': LABELS,
    'Trees Averaged': trees_per_label,
    'Verification Samples': 100,
    'Maximum Averaging Error': averaging_errors,
    'Verification': 'Passed'
})

averaging_verification_df.to_csv(
    RUN_TABLES_DIR / 'probability_averaging_verification.csv',
    index=False
)

print("Probability averaging: Verified")
print("Training score shape:", train_scores.shape)
print("Validation score shape:", val_scores.shape)

display(averaging_verification_df)


# Inspect validation probability resolution

score_distribution_records = []

for i, label in enumerate(LABELS):

    scores = val_scores[:, i]

    unique_scores, counts = np.unique(
        scores,
        return_counts=True
    )

    score_distribution_records.append({
        'Label': label,
        'Minimum Score': float(scores.min()),
        'Maximum Score': float(scores.max()),
        'Mean Score': float(scores.mean()),
        'Standard Deviation': float(scores.std()),
        'Median Score': float(np.median(scores)),
        'Unique Scores': len(unique_scores),
        'Exact Zero Scores': int((scores == 0).sum()),
        'Exact One Scores': int((scores == 1).sum()),
        'Exact Default Threshold Scores':
            int((scores == DEFAULT_THRESHOLD).sum()),
        'Most Frequent Score':
            float(unique_scores[np.argmax(counts)]),
        'Most Frequent Score Count':
            int(counts.max()),
        'Most Frequent Score Share (%)':
            float(100 * counts.max() / len(scores))
    })

score_distribution_df = pd.DataFrame(
    score_distribution_records
)

score_distribution_df.to_csv(
    RUN_TABLES_DIR / 'probability_score_distribution.csv',
    index=False
)

print("\nValidation probability score distribution")

display(score_distribution_df.round(6))


# Evaluate default operating points

train_default_metrics, train_default_per_label = (
    evaluate_multilabel(
        y_train,
        train_scores,
        LABELS,
        thresholds=DEFAULT_THRESHOLD
    )
)

default_metrics, default_per_label_df = (
    evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=DEFAULT_THRESHOLD
    )
)


# Optimize thresholds using validation data only

threshold_df = optimize_thresholds(
    y_val,
    val_scores,
    LABELS
)

assert threshold_df['Label'].tolist() == LABELS

optimized_thresholds = threshold_df[
    'Optimal Threshold'
].to_numpy(dtype=np.float64)

assert optimized_thresholds.shape == (len(LABELS),)
assert np.isfinite(optimized_thresholds).all()
assert (optimized_thresholds >= 0).all()
assert (optimized_thresholds <= 1).all()

optimized_metrics, optimized_per_label_df = (
    evaluate_multilabel(
        y_val,
        val_scores,
        LABELS,
        thresholds=optimized_thresholds
    )
)

assert np.all(
    optimized_per_label_df['F1'].to_numpy()
    >= default_per_label_df['F1'].to_numpy() - 1e-12
)

np.testing.assert_allclose(
    optimized_per_label_df['F1'].to_numpy(),
    threshold_df['Validation F1'].to_numpy(),
    rtol=0,
    atol=1e-12
)

for metric in [
    'Macro AP',
    'Micro AP',
    'Macro ROC-AUC',
    'Micro ROC-AUC'
]:

    assert np.isclose(
        default_metrics[metric],
        optimized_metrics[metric],
        rtol=0,
        atol=1e-12
    )


# Apply validation-selected thresholds to training scores

train_optimized_metrics, train_optimized_per_label = (
    evaluate_multilabel(
        y_train,
        train_scores,
        LABELS,
        thresholds=optimized_thresholds
    )
)


# Compare overall validation performance

overall_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Default': default_metrics[metric],
        'Optimized': optimized_metrics[metric],
        'Change':
            optimized_metrics[metric] - default_metrics[metric]
    }
    for metric in default_metrics
])

print("\nOverall validation performance")

display(overall_comparison_df.round(6))

overall_comparison_df.to_csv(
    RUN_TABLES_DIR / 'overall_threshold_comparison.csv',
    index=False
)


# Compare per-label threshold effects

per_label_comparison_df = pd.DataFrame({
    'Label': LABELS,
    'Average Precision':
        optimized_per_label_df['Average Precision'],
    'ROC-AUC':
        optimized_per_label_df['ROC-AUC'],
    'Default Threshold': DEFAULT_THRESHOLD,
    'Optimized Threshold': optimized_thresholds,
    'Default Precision':
        default_per_label_df['Precision'],
    'Optimized Precision':
        optimized_per_label_df['Precision'],
    'Default Recall':
        default_per_label_df['Recall'],
    'Optimized Recall':
        optimized_per_label_df['Recall'],
    'Default F1':
        default_per_label_df['F1'],
    'Optimized F1':
        optimized_per_label_df['F1'],
    'Actual Positives':
        optimized_per_label_df['Positive Support']
})

per_label_comparison_df['F1 Improvement'] = (
    per_label_comparison_df['Optimized F1']
    - per_label_comparison_df['Default F1']
)

print("\nPer-label validation performance")

display(per_label_comparison_df.round(6))

per_label_comparison_df.to_csv(
    RUN_TABLES_DIR / 'per_label_threshold_comparison.csv',
    index=False
)


# Analyze training-validation generalization gaps

generalization_records = []

for threshold_type, train_metrics, val_metrics in [
    (
        'Default',
        train_default_metrics,
        default_metrics
    ),
    (
        'Validation-optimized',
        train_optimized_metrics,
        optimized_metrics
    )
]:

    for metric in [
        'Macro AP',
        'Macro ROC-AUC',
        'Macro Precision',
        'Macro Recall',
        'Macro F1',
        'Micro F1'
    ]:

        generalization_records.append({
            'Threshold Type': threshold_type,
            'Metric': metric,
            'Training': train_metrics[metric],
            'Validation': val_metrics[metric],
            'Training-Validation Gap':
                train_metrics[metric] - val_metrics[metric]
        })

generalization_df = pd.DataFrame(
    generalization_records
)

print("\nTraining-validation generalization comparison")

display(generalization_df.round(6))

generalization_df.to_csv(
    RUN_TABLES_DIR / 'training_validation_comparison.csv',
    index=False
)


# Measure individual label generalization gaps

per_label_generalization_df = pd.DataFrame({
    'Label': LABELS,
    'Training AP':
        train_default_per_label['Average Precision'],
    'Validation AP':
        default_per_label_df['Average Precision'],
    'Training Default F1':
        train_default_per_label['F1'],
    'Validation Default F1':
        default_per_label_df['F1'],
    'Training F1 at Validation Threshold':
        train_optimized_per_label['F1'],
    'Validation Optimized F1':
        optimized_per_label_df['F1']
})

per_label_generalization_df['AP Gap'] = (
    per_label_generalization_df['Training AP']
    - per_label_generalization_df['Validation AP']
)

per_label_generalization_df['Default F1 Gap'] = (
    per_label_generalization_df['Training Default F1']
    - per_label_generalization_df['Validation Default F1']
)

per_label_generalization_df['Optimized F1 Gap'] = (
    per_label_generalization_df[
        'Training F1 at Validation Threshold'
    ]
    - per_label_generalization_df[
        'Validation Optimized F1'
    ]
)

print("\nPer-label generalization gaps")

display(per_label_generalization_df.round(6))

per_label_generalization_df.to_csv(
    RUN_TABLES_DIR / 'per_label_generalization_comparison.csv',
    index=False
)


# Save complete evaluation tables

for filename, metrics in [
    ('default_overall_metrics.csv', default_metrics),
    ('optimized_overall_metrics.csv', optimized_metrics),
    ('training_default_overall_metrics.csv', train_default_metrics),
    ('training_optimized_overall_metrics.csv', train_optimized_metrics)
]:

    pd.DataFrame({
        'Metric': list(metrics.keys()),
        'Value': list(metrics.values())
    }).to_csv(
        RUN_TABLES_DIR / filename,
        index=False
    )

for filename, table in [
    ('default_per_label_metrics.csv', default_per_label_df),
    ('optimized_per_label_metrics.csv', optimized_per_label_df),
    ('training_default_per_label_metrics.csv', train_default_per_label),
    ('training_optimized_per_label_metrics.csv', train_optimized_per_label),
    ('optimized_thresholds.csv', threshold_df),
    ('feature_verification.csv', feature_verification_df)
]:

    table.to_csv(
        RUN_TABLES_DIR / filename,
        index=False
    )


# Save ID-aligned validation predictions

default_predictions = apply_thresholds(
    val_scores,
    DEFAULT_THRESHOLD
)

optimized_predictions = apply_thresholds(
    val_scores,
    optimized_thresholds
)

predictions_df = pd.DataFrame({
    'id': val_df['id'].to_numpy()
})

for i, label in enumerate(LABELS):

    predictions_df[f'{label}_true'] = y_val[:, i]

    predictions_df[f'{label}_score'] = val_scores[:, i]

    predictions_df[f'{label}_pred_default'] = (
        default_predictions[:, i].astype(np.int8)
    )

    predictions_df[f'{label}_pred_optimized'] = (
        optimized_predictions[:, i].astype(np.int8)
    )

assert predictions_df.shape == (
    len(y_val),
    1 + 4 * len(LABELS)
)

PREDICTIONS_PATH = (
    RUN_PREDICTIONS_DIR / 'validation_predictions.parquet'
)

predictions_df.to_parquet(
    PREDICTIONS_PATH,
    index=False
)

saved_predictions_df = pd.read_parquet(
    PREDICTIONS_PATH
)

pd.testing.assert_frame_equal(
    predictions_df,
    saved_predictions_df,
    check_exact=True
)


# Save full-precision threshold configuration

threshold_config = {
    'run_id': RUN_ID,
    'labels': LABELS,
    'score_type': SCORE_TYPE,
    'default_threshold': DEFAULT_THRESHOLD,
    'optimized_thresholds': {
        label: float(threshold)
        for label, threshold in zip(
            LABELS,
            optimized_thresholds
        )
    },
    'selection_partition': 'validation',
    'optimization_metric': 'per_label_f1',
    'tie_breaking': 'highest_threshold',
    'comparison_operator': '>=',
    'probability_calibration': False,
    'prediction_parallel_jobs_per_forest': 1,
    'official_test_used': False
}

THRESHOLD_PATH = (
    RUN_CONFIGS_DIR / f'{RUN_ID}_thresholds.json'
)

with open(THRESHOLD_PATH, 'w') as file:
    json.dump(threshold_config, file, indent=4)

with open(THRESHOLD_PATH, 'r') as file:
    saved_threshold_config = json.load(file)

assert saved_threshold_config == threshold_config

reloaded_thresholds = np.array([
    saved_threshold_config['optimized_thresholds'][label]
    for label in LABELS
], dtype=np.float64)

assert np.array_equal(
    optimized_thresholds,
    reloaded_thresholds
)


# Reload the saved model and reconstruct validation scores

assert calculate_sha256(MODEL_PATH) == model_hash

reloaded_rf_model = joblib.load(MODEL_PATH)

reconstructed_scores = predict_rf_probabilities(
    reloaded_rf_model,
    X_val
)

maximum_reconstruction_error = float(
    np.abs(
        val_scores - reconstructed_scores
    ).max()
)

np.testing.assert_allclose(
    val_scores,
    reconstructed_scores,
    rtol=0,
    atol=1e-12
)

assert np.array_equal(
    default_predictions,
    apply_thresholds(
        reconstructed_scores,
        DEFAULT_THRESHOLD
    )
)

assert np.array_equal(
    optimized_predictions,
    apply_thresholds(
        reconstructed_scores,
        reloaded_thresholds
    )
)


# Verify metrics from persisted prediction artifacts

saved_targets = saved_predictions_df[[
    f'{label}_true'
    for label in LABELS
]].to_numpy()

saved_scores = saved_predictions_df[[
    f'{label}_score'
    for label in LABELS
]].to_numpy(dtype=np.float64)

assert saved_predictions_df['id'].tolist() == val_df['id'].tolist()

assert np.array_equal(saved_targets, y_val)
assert np.array_equal(saved_scores, val_scores)

for threshold_values, original_metrics, original_per_label in [
    (
        DEFAULT_THRESHOLD,
        default_metrics,
        default_per_label_df
    ),
    (
        reloaded_thresholds,
        optimized_metrics,
        optimized_per_label_df
    )
]:

    reconstructed_metrics, reconstructed_per_label = (
        evaluate_multilabel(
            saved_targets,
            saved_scores,
            LABELS,
            thresholds=threshold_values
        )
    )

    for metric in original_metrics:

        assert np.isclose(
            original_metrics[metric],
            reconstructed_metrics[metric],
            rtol=0,
            atol=1e-12
        )

    pd.testing.assert_frame_equal(
        original_per_label,
        reconstructed_per_label,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )


# Save the completed run configuration

run_config = {
    **rf_experiment_config,
    'feature_count': FEATURE_BUDGET,
    'prediction_parallel_jobs_per_forest': 1,
    'prediction_averaging_strategy': 'sequential',
    'model_path': str(
        MODEL_PATH.relative_to(PROJECT_ROOT)
    ),
    'threshold_path': str(
        THRESHOLD_PATH.relative_to(PROJECT_ROOT)
    ),
    'predictions_path': str(
        PREDICTIONS_PATH.relative_to(PROJECT_ROOT)
    )
}

RUN_CONFIG_PATH = (
    RUN_CONFIGS_DIR / f'{RUN_ID}.json'
)

with open(RUN_CONFIG_PATH, 'w') as file:
    json.dump(run_config, file, indent=4)

with open(RUN_CONFIG_PATH, 'r') as file:
    assert json.load(file) == run_config


# Construct the experiment summary

run_summary = {
    'Run ID': RUN_ID,
    'Model': MODEL_NAME,
    'Feature Representation': REPRESENTATION,
    'Class Weight': 'None',
    'Training Samples': len(y_train),
    'Validation Samples': len(y_val),
    'Features': FEATURE_BUDGET,
    'Macro AP': optimized_metrics['Macro AP'],
    'Micro AP': optimized_metrics['Micro AP'],
    'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
    'Default Macro F1': default_metrics['Macro F1'],
    'Optimized Macro F1': optimized_metrics['Macro F1'],
    'Default Micro F1': default_metrics['Micro F1'],
    'Optimized Micro F1': optimized_metrics['Micro F1'],
    'Optimized Macro Precision':
        optimized_metrics['Macro Precision'],
    'Optimized Macro Recall':
        optimized_metrics['Macro Recall'],
    'Training Macro AP': train_default_metrics['Macro AP'],
    'Training Default Macro F1':
        train_default_metrics['Macro F1'],
    'Macro AP Generalization Gap': (
        train_default_metrics['Macro AP']
        - default_metrics['Macro AP']
    ),
    'Training Time (s)': training_time,
    'Model Fit Time (s)': model_fit_time,
    'Validation Prediction Time (s)':
        validation_prediction_time,
    'Training Prediction Time (s)':
        training_prediction_time,
    'Feature Loading Time (s)': feature_loading_time,
    'Model Size (MiB)': model_size_mib,
    'Binary Forests': len(rf_model.estimators_),
    'Trees per Forest': trees_per_label,
    'Total Trees': len(rf_tree_structure_df),
    'Maximum Observed Depth':
        int(rf_tree_structure_df['Actual Depth'].max()),
    'Total Leaves':
        int(rf_tree_structure_df['Leaf Count'].sum()),
    'Total Nodes':
        int(rf_tree_structure_df['Node Count'].sum()),
    'Sampled Peak Training Memory (MiB)':
        sampled_peak_memory_mib,
    'Sampled Peak Memory Increase (MiB)':
        memory_summary['sampled_peak_increase_mib'],
    'Training Warnings': len(training_warning_records),
    'Maximum Score Reconstruction Error':
        maximum_reconstruction_error,
    'Structural Verification': 'Passed',
    'Official Test Used': False,
    'Status': 'Completed'
}

run_summary_df = pd.DataFrame([run_summary])

run_summary_df.to_csv(
    RUN_TABLES_DIR / 'run_summary.csv',
    index=False
)

print("\nRandom Forest experiment summary")

display(run_summary_df.T)


# Update the training and evaluation log

evaluation_end_time = datetime.now(
    timezone.utc
).isoformat()

training_log.update({
    'validation_prediction_time_seconds':
        float(validation_prediction_time),
    'training_prediction_time_seconds':
        float(training_prediction_time),
    'prediction_parallel_jobs_per_forest': 1,
    'evaluation_end_time': evaluation_end_time,
    'maximum_probability_averaging_error':
        max(averaging_errors),
    'maximum_score_reconstruction_error':
        maximum_reconstruction_error,
    'probability_averaging_verified': True,
    'score_alignment_verified': True,
    'model_reconstruction_verified': True,
    'metric_reconstruction_verified': True,
    'prediction_roundtrip_verified': True,
    'threshold_roundtrip_verified': True,
    'validation_evaluation_completed': True,
    'threshold_optimization_completed': True,
    'training_evaluation_completed': True,
    'official_test_used': False,
    'status': 'Completed'
})

with open(TRAINING_LOG_PATH, 'w') as file:
    json.dump(training_log, file, indent=4)


# Fingerprint critical experiment artifacts

critical_artifacts = [
    ('Model', MODEL_PATH),
    ('Run Configuration', RUN_CONFIG_PATH),
    ('Threshold Configuration', THRESHOLD_PATH),
    ('Validation Predictions', PREDICTIONS_PATH),
    ('Training Log', TRAINING_LOG_PATH),
    ('Feature Manifest',
     FEATURES_DIR / 'feature_artifact_manifest.csv'),
    ('Evaluation Module',
     SRC_DIR / 'evaluation_utils.py')
]

artifact_records = []

for artifact_type, path in critical_artifacts:

    assert path.is_file()
    assert path.stat().st_size > 0

    artifact_records.append({
        'Run ID': RUN_ID,
        'Artifact Type': artifact_type,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size,
        'SHA256': calculate_sha256(path)
    })

artifact_manifest_df = pd.DataFrame(
    artifact_records
)

artifact_manifest_df.to_csv(
    RUN_TABLES_DIR / 'artifact_manifest.csv',
    index=False
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Store the completed experiment for comparative analysis

rf_results[RUN_ID] = {
    'spec': experiment_spec.copy(),
    'summary': run_summary,
    'default_metrics': default_metrics,
    'optimized_metrics': optimized_metrics,
    'default_per_label': default_per_label_df,
    'optimized_per_label': optimized_per_label_df,
    'thresholds': threshold_df,
    'model_path': MODEL_PATH,
    'config_path': RUN_CONFIG_PATH,
    'threshold_path': THRESHOLD_PATH,
    'predictions_path': PREDICTIONS_PATH,
    'results_path': RUN_DIR
}

del reloaded_rf_model
del reconstructed_scores
del saved_predictions_df

gc.collect()


# Summarize evaluation performance

print("\nRandom Forest evaluation summary")

print("Training Macro AP:",
      round(train_default_metrics['Macro AP'], 6))

print("Validation Macro AP:",
      round(default_metrics['Macro AP'], 6))

print("Macro AP generalization gap:",
      round(run_summary['Macro AP Generalization Gap'], 6))

print("Training default Macro F1:",
      round(train_default_metrics['Macro F1'], 6))

print("Validation default Macro F1:",
      round(default_metrics['Macro F1'], 6))

print("Validation optimized Macro F1:",
      round(optimized_metrics['Macro F1'], 6))

print("Validation optimized Micro F1:",
      round(optimized_metrics['Micro F1'], 6))

print("Validation Macro ROC-AUC:",
      round(optimized_metrics['Macro ROC-AUC'], 6))

print("Macro F1 improvement:",
      round(
          optimized_metrics['Macro F1']
          - default_metrics['Macro F1'],
          6
      ))

print("Labels with improved F1:",
      int(
          (per_label_comparison_df['F1 Improvement'] > 0).sum()
      ),
      "/", len(LABELS))

print("Validation prediction time (s):",
      round(validation_prediction_time, 6))

print("Maximum probability averaging error:",
      max(averaging_errors))

print("Maximum score reconstruction error:",
      maximum_reconstruction_error)

print("Probability averaging: Passed")
print("Model reconstruction: Passed")
print("Metric reconstruction: Passed")
print("Threshold roundtrip: Passed")
print("Prediction roundtrip: Passed")

print("Critical artifacts fingerprinted:",
      len(artifact_manifest_df))

print("Completed Random Forest experiments:",
      len(rf_results))

print("Experiment registration: Pending")
print("Official test dataset: Not loaded")

print("\nRandom Forest evaluation completed successfully.")

In [ ]:
# Define the five common-feature baseline experiments

baseline_run_ids = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'SVM_WORD_BASELINE_S42',
    'DT_WORD_BASELINE_S42',
    RUN_ID
]

baseline_results = {}
baseline_summary_records = []
baseline_per_label_records = []

for run_id in baseline_run_ids:

    if run_id == RUN_ID:

        current_config = run_config
        model_name = MODEL_NAME
        model_path = MODEL_PATH
        predictions_path = PREDICTIONS_PATH
        fit_time = training_time

        expected_ap = run_summary['Macro AP']
        expected_f1 = run_summary['Optimized Macro F1']

    else:

        record = registry_df.loc[
            registry_df['run_id'] == run_id
        ].iloc[0]

        current_config = previous_run_configs[run_id]

        model_name = record['model']
        model_path = PROJECT_ROOT / record['model_path']

        previous_run_dir = (
            PROJECT_ROOT / record['results_path']
        )

        predictions_dir = previous_run_dir / 'predictions'

        candidate_paths = [
            predictions_dir / 'validation_predictions.parquet',
            predictions_dir / f'{run_id}_validation_predictions.parquet'
        ]

        available_paths = [
            path for path in candidate_paths
            if path.is_file()
        ]

        assert len(available_paths) == 1

        predictions_path = available_paths[0]

        fit_time = float(record['training_time_seconds'])
        expected_ap = float(record['validation_macro_ap'])
        expected_f1 = float(record['validation_macro_f1'])


    # Verify identical experimental conditions

    assert current_config['feature_representation'] == REPRESENTATION

    assert current_config['training_dataset_sha256'] == (
        EXPECTED_TRAINING_HASH
    )

    assert current_config['validation_dataset_sha256'] == (
        EXPECTED_VALIDATION_HASH
    )

    assert current_config['evaluation_module_sha256'] == (
        evaluation_hash
    )

    assert current_config['feature_manifest_sha256'] == (
        feature_manifest_hash
    )

    assert current_config['labels'] == LABELS
    assert current_config['seed'] == SEED
    assert not current_config['official_test_used']

    assert model_path.is_file()
    assert predictions_path.is_file()


    # Reload and verify ID-aligned validation predictions

    baseline_predictions = pd.read_parquet(
        predictions_path
    )

    assert baseline_predictions['id'].is_unique

    assert (
        baseline_predictions['id'].tolist()
        == val_df['id'].tolist()
    )

    targets = baseline_predictions[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    scores = baseline_predictions[[
        f'{label}_score' for label in LABELS
    ]].to_numpy(dtype=np.float64)

    assert np.array_equal(targets, y_val)
    assert scores.shape == y_val.shape
    assert np.isfinite(scores).all()

    default_threshold = (
        0.0 if model_name == 'Linear SVM'
        else 0.5
    )

    if model_name != 'Linear SVM':

        assert (scores >= 0).all()
        assert (scores <= 1).all()


    # Reconstruct default and optimized metrics

    baseline_default_metrics, baseline_default_labels = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=default_threshold
        )
    )

    baseline_threshold_df = optimize_thresholds(
        targets,
        scores,
        LABELS
    )

    baseline_thresholds = baseline_threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    baseline_optimized_metrics, baseline_optimized_labels = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=baseline_thresholds
        )
    )


    # Verify reconstructed binary predictions

    saved_default = baseline_predictions[[
        f'{label}_pred_default' for label in LABELS
    ]].to_numpy()

    saved_optimized = baseline_predictions[[
        f'{label}_pred_optimized' for label in LABELS
    ]].to_numpy()

    assert np.array_equal(
        saved_default,
        apply_thresholds(scores, default_threshold)
    )

    assert np.array_equal(
        saved_optimized,
        apply_thresholds(scores, baseline_thresholds)
    )

    assert np.isclose(
        baseline_optimized_metrics['Macro AP'],
        expected_ap,
        rtol=0,
        atol=1e-6
    )

    assert np.isclose(
        baseline_optimized_metrics['Macro F1'],
        expected_f1,
        rtol=0,
        atol=1e-6
    )


    # Record reconstructed overall performance

    baseline_summary_records.append({
        'Run ID': run_id,
        'Model': model_name,
        'Representation': REPRESENTATION,
        'Training Samples': len(y_train),
        'Validation Samples': len(y_val),
        'Features': FEATURE_BUDGET,
        'Macro AP': baseline_optimized_metrics['Macro AP'],
        'Macro ROC-AUC':
            baseline_optimized_metrics['Macro ROC-AUC'],
        'Default Macro F1':
            baseline_default_metrics['Macro F1'],
        'Optimized Macro F1':
            baseline_optimized_metrics['Macro F1'],
        'Default Micro F1':
            baseline_default_metrics['Micro F1'],
        'Optimized Micro F1':
            baseline_optimized_metrics['Micro F1'],
        'Optimized Macro Precision':
            baseline_optimized_metrics['Macro Precision'],
        'Optimized Macro Recall':
            baseline_optimized_metrics['Macro Recall'],
        'Optimized Hamming Loss':
            baseline_optimized_metrics['Hamming Loss'],
        'Optimized Exact Match Accuracy':
            baseline_optimized_metrics['Exact Match Accuracy'],
        'Training Time (s)': fit_time,
        'Model Size (MiB)':
            model_path.stat().st_size / (1024 ** 2),
        'Metric Reconstruction': 'Passed',
        'Official Test Used': False
    })

    label_metrics = baseline_optimized_labels.copy()

    label_metrics.insert(0, 'Model', model_name)
    label_metrics.insert(0, 'Run ID', run_id)

    label_metrics['Default F1'] = (
        baseline_default_labels['F1'].to_numpy()
    )

    baseline_per_label_records.append(label_metrics)

    baseline_results[run_id] = {
        'model': model_name,
        'default_metrics': baseline_default_metrics,
        'optimized_metrics': baseline_optimized_metrics,
        'default_per_label': baseline_default_labels,
        'optimized_per_label': baseline_optimized_labels
    }

    print(f'{run_id}: Reconstruction passed')


# Consolidate the five common-feature baselines

rf_baseline_comparison_df = pd.DataFrame(
    baseline_summary_records
)

assert len(rf_baseline_comparison_df) == 5

assert rf_baseline_comparison_df[
    'Metric Reconstruction'
].eq('Passed').all()

rf_baseline_comparison_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_comparison.csv',
    index=False
)

rf_baseline_per_label_df = pd.concat(
    baseline_per_label_records,
    ignore_index=True
)

assert len(rf_baseline_per_label_df) == 5 * len(LABELS)

rf_baseline_per_label_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_per_label.csv',
    index=False
)


# Rank models using validation Macro AP

rf_baseline_ranking_df = (
    rf_baseline_comparison_df.sort_values(
        ['Macro AP', 'Optimized Macro F1', 'Run ID'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

rf_baseline_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(rf_baseline_ranking_df) + 1)
)

rf_baseline_ranking_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_ranking.csv',
    index=False
)

print("\nCommon Word TF-IDF baseline ranking")

display(
    rf_baseline_ranking_df[[
        'Rank',
        'Model',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)',
        'Model Size (MiB)'
    ]].round(6)
)


# Generate per-label comparison tables

model_order = rf_baseline_comparison_df[
    'Model'
].tolist()

ap_comparison_df = (
    rf_baseline_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='Average Precision'
    )
    .reindex(index=LABELS, columns=model_order)
)

f1_comparison_df = (
    rf_baseline_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='F1'
    )
    .reindex(index=LABELS, columns=model_order)
)

assert not ap_comparison_df.isna().any().any()
assert not f1_comparison_df.isna().any().any()

ap_comparison_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_ap_comparison.csv'
)

f1_comparison_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_f1_comparison.csv'
)

print("\nPer-label Average Precision comparison")

display(ap_comparison_df.round(6))

print("\nPer-label optimized F1 comparison")

display(f1_comparison_df.round(6))


# Reload verified Decision Tree diagnostic artifacts

DT_TABLES_DIR = DT_RUN_DIR / 'tables'

dt_generalization_df = pd.read_csv(
    DT_TABLES_DIR / 'per_label_generalization_comparison.csv'
)

dt_score_distribution_df = pd.read_csv(
    DT_TABLES_DIR / 'probability_score_distribution.csv'
)

dt_structure_reference_df = pd.read_csv(
    DT_TABLES_DIR / 'classifier_structure_summary.csv'
)

assert dt_generalization_df['Label'].tolist() == LABELS
assert dt_score_distribution_df['Label'].tolist() == LABELS
assert dt_structure_reference_df['Label'].tolist() == LABELS

dt_reference = baseline_results[
    'DT_WORD_BASELINE_S42'
]

np.testing.assert_allclose(
    dt_generalization_df['Validation AP'],
    dt_reference['optimized_per_label']['Average Precision'],
    rtol=0,
    atol=1e-12
)


# Compare overall Decision Tree and Random Forest performance

comparison_metrics = [
    'Macro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Training Macro AP',
    'Macro AP Generalization Gap',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

rf_dt_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Decision Tree': float(dt_summary[metric]),
        'Random Forest': float(run_summary[metric]),
        'RF minus DT': (
            float(run_summary[metric])
            - float(dt_summary[metric])
        )
    }
    for metric in comparison_metrics
])

rf_dt_comparison_df.to_csv(
    TABLES_DIR / 'rf_dt_overall_comparison.csv',
    index=False
)

print("\nDecision Tree versus Random Forest")

display(rf_dt_comparison_df.round(6))


# Compare per-label predictive performance and generalization

rf_dt_per_label_df = pd.DataFrame({
    'Label': LABELS,
    'DT Validation AP':
        dt_generalization_df['Validation AP'],
    'RF Validation AP':
        per_label_generalization_df['Validation AP'],
    'DT Optimized F1':
        dt_generalization_df['Validation Optimized F1'],
    'RF Optimized F1':
        per_label_generalization_df['Validation Optimized F1'],
    'DT Training AP':
        dt_generalization_df['Training AP'],
    'RF Training AP':
        per_label_generalization_df['Training AP'],
    'DT AP Gap':
        dt_generalization_df['AP Gap'],
    'RF AP Gap':
        per_label_generalization_df['AP Gap'],
    'DT Unique Scores':
        dt_score_distribution_df['Unique Scores'],
    'RF Unique Scores':
        score_distribution_df['Unique Scores'],
    'DT Most Frequent Score Share (%)':
        dt_score_distribution_df[
            'Most Frequent Score Share (%)'
        ],
    'RF Most Frequent Score Share (%)':
        score_distribution_df[
            'Most Frequent Score Share (%)'
        ]
})

rf_dt_per_label_df['AP Change'] = (
    rf_dt_per_label_df['RF Validation AP']
    - rf_dt_per_label_df['DT Validation AP']
)

rf_dt_per_label_df['Optimized F1 Change'] = (
    rf_dt_per_label_df['RF Optimized F1']
    - rf_dt_per_label_df['DT Optimized F1']
)

rf_dt_per_label_df['AP Gap Reduction'] = (
    rf_dt_per_label_df['DT AP Gap']
    - rf_dt_per_label_df['RF AP Gap']
)

rf_dt_per_label_df['Score Concentration Reduction (pp)'] = (
    rf_dt_per_label_df[
        'DT Most Frequent Score Share (%)'
    ]
    - rf_dt_per_label_df[
        'RF Most Frequent Score Share (%)'
    ]
)

rf_dt_per_label_df.to_csv(
    TABLES_DIR / 'rf_dt_per_label_comparison.csv',
    index=False
)

print("\nPer-label Decision Tree versus Random Forest diagnostics")

display(rf_dt_per_label_df.round(6))


# Compare rare-label performance

rare_label_comparison_df = rf_dt_per_label_df.loc[
    rf_dt_per_label_df['Label'].isin(rare_labels)
].copy()

rare_label_comparison_df.to_csv(
    TABLES_DIR / 'rf_dt_rare_label_comparison.csv',
    index=False
)

print("\nRare-label comparison")

display(
    rare_label_comparison_df[[
        'Label',
        'DT Validation AP',
        'RF Validation AP',
        'AP Change',
        'DT Optimized F1',
        'RF Optimized F1',
        'Optimized F1 Change',
        'AP Gap Reduction'
    ]].round(6)
)


# Record threshold sensitivity across all five baselines

threshold_sensitivity_df = rf_baseline_comparison_df[[
    'Run ID',
    'Model',
    'Macro AP',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1'
]].copy()

threshold_sensitivity_df['Macro F1 Improvement'] = (
    threshold_sensitivity_df['Optimized Macro F1']
    - threshold_sensitivity_df['Default Macro F1']
)

threshold_sensitivity_df['Micro F1 Improvement'] = (
    threshold_sensitivity_df['Optimized Micro F1']
    - threshold_sensitivity_df['Default Micro F1']
)

threshold_sensitivity_df.to_csv(
    TABLES_DIR / 'rf_common_baseline_threshold_sensitivity.csv',
    index=False
)

print("\nThreshold sensitivity across common-feature baselines")

display(threshold_sensitivity_df.round(6))


# Figure 1: Threshold sensitivity across the five algorithms

short_model_names = {
    'Multinomial Naive Bayes': 'Naive Bayes',
    'Logistic Regression': 'Logistic Regression',
    'Linear SVM': 'Linear SVM',
    'Decision Tree': 'Decision Tree',
    'Random Forest': 'Random Forest'
}

plot_df = threshold_sensitivity_df.set_index(
    'Model'
).loc[model_order].copy()

plot_df.index = [
    short_model_names[model]
    for model in plot_df.index
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6)
)

for ax, columns, title in [
    (
        axes[0],
        ['Default Macro F1', 'Optimized Macro F1'],
        'Macro F1'
    ),
    (
        axes[1],
        ['Default Micro F1', 'Optimized Micro F1'],
        'Micro F1'
    )
]:

    plot_df[columns].plot(
        kind='bar',
        ax=ax,
        color=['#94A3B8', '#4C78A8'],
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt='%.3f',
            padding=3,
            fontsize=9
        )

    ax.set_title(title)
    ax.set_ylabel('F1 score')
    ax.set_xlabel('')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='x', visible=False)
    ax.legend(['Default threshold', 'Optimized thresholds'])

fig.suptitle(
    'Threshold Sensitivity of Common Word TF-IDF Baselines',
    fontsize=16
)

sns.despine()

fig.tight_layout(rect=[0, 0, 1, 0.94])

fig.savefig(
    FIGURES_DIR / 'rf_common_baseline_threshold_sensitivity.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Figure 2: Decision Tree versus Random Forest diagnostics

diagnostic_plot_df = rf_dt_per_label_df.set_index(
    'Label'
).loc[LABELS].copy()

diagnostic_plot_df.index = [
    label.replace('_', ' ').title()
    for label in LABELS
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11)
)

diagnostic_panels = [
    (
        axes[0, 0],
        ['DT Validation AP', 'RF Validation AP'],
        'Validation Average Precision',
        'Average Precision',
        1.05,
        '%.3f'
    ),
    (
        axes[0, 1],
        ['DT Optimized F1', 'RF Optimized F1'],
        'Validation Optimized F1',
        'F1 score',
        1.05,
        '%.3f'
    ),
    (
        axes[1, 0],
        ['DT AP Gap', 'RF AP Gap'],
        'Training-Validation AP Generalization Gap',
        'AP gap',
        None,
        '%.3f'
    ),
    (
        axes[1, 1],
        [
            'DT Most Frequent Score Share (%)',
            'RF Most Frequent Score Share (%)'
        ],
        'Probability Score Concentration',
        'Comments receiving most frequent score (%)',
        112,
        '%.1f%%'
    )
]

for ax, columns, title, ylabel, upper_limit, fmt in diagnostic_panels:

    diagnostic_plot_df[columns].plot(
        kind='bar',
        ax=ax,
        color=[
            MODEL_COLORS['Decision Tree'],
            MODEL_COLORS['Random Forest']
        ],
        width=0.7
    )

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt=fmt,
            padding=3,
            fontsize=8
        )

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')

    if upper_limit is not None:

        ax.set_ylim(0, upper_limit)

    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='x', visible=False)
    ax.legend(['Decision Tree', 'Random Forest'])

fig.suptitle(
    'Decision Tree versus Random Forest: Predictive Diagnostics',
    fontsize=16
)

sns.despine()

fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'rf_dt_predictive_diagnostics.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Calculate evidence-based comparative findings

rf_rank = int(
    rf_baseline_ranking_df.loc[
        rf_baseline_ranking_df['Run ID'] == RUN_ID,
        'Rank'
    ].iloc[0]
)

macro_ap_improvement = (
    run_summary['Macro AP']
    - float(dt_summary['Macro AP'])
)

macro_f1_improvement = (
    run_summary['Optimized Macro F1']
    - float(dt_summary['Optimized Macro F1'])
)

dt_ap_gap = float(
    dt_summary['Macro AP Generalization Gap']
)

rf_ap_gap = float(
    run_summary['Macro AP Generalization Gap']
)

ap_gap_reduction = dt_ap_gap - rf_ap_gap

ap_gap_reduction_percentage = (
    100 * ap_gap_reduction / dt_ap_gap
)

training_speed_ratio = (
    float(dt_summary['Training Time (s)'])
    / training_time
)

ap_improved_labels = int(
    (rf_dt_per_label_df['AP Change'] > 0).sum()
)

f1_improved_labels = int(
    (rf_dt_per_label_df['Optimized F1 Change'] > 0).sum()
)

gap_reduced_labels = int(
    (rf_dt_per_label_df['AP Gap Reduction'] > 0).sum()
)

zero_default_prediction_labels = (
    default_per_label_df.loc[
        default_per_label_df['Predicted Positives'] == 0,
        'Label'
    ].tolist()
)

largest_gap_row = rf_dt_per_label_df.loc[
    rf_dt_per_label_df['RF AP Gap'].idxmax()
]

highest_concentration_row = rf_dt_per_label_df.loc[
    rf_dt_per_label_df[
        'RF Most Frequent Score Share (%)'
    ].idxmax()
]


# Save comparative findings and limitations

rf_analysis_config = {
    'run_id': RUN_ID,
    'comparison_scope': 'common_word_tfidf_baselines',
    'models_compared': len(baseline_results),
    'common_baseline_rank': rf_rank,
    'primary_selection_metric': 'macro_average_precision',
    'validation_macro_ap': float(run_summary['Macro AP']),
    'validation_optimized_macro_f1':
        float(run_summary['Optimized Macro F1']),
    'reference_decision_tree_run': 'DT_WORD_BASELINE_S42',
    'macro_ap_improvement_vs_dt':
        float(macro_ap_improvement),
    'optimized_macro_f1_improvement_vs_dt':
        float(macro_f1_improvement),
    'decision_tree_ap_generalization_gap': dt_ap_gap,
    'random_forest_ap_generalization_gap': rf_ap_gap,
    'absolute_ap_gap_reduction':
        float(ap_gap_reduction),
    'relative_ap_gap_reduction_percentage':
        float(ap_gap_reduction_percentage),
    'labels_with_ap_improvement_vs_dt':
        ap_improved_labels,
    'labels_with_f1_improvement_vs_dt':
        f1_improved_labels,
    'labels_with_reduced_ap_gap':
        gap_reduced_labels,
    'labels_with_no_default_positive_predictions':
        zero_default_prediction_labels,
    'largest_rf_ap_gap_label':
        str(largest_gap_row['Label']),
    'largest_rf_ap_gap':
        float(largest_gap_row['RF AP Gap']),
    'highest_rf_score_concentration_label':
        str(highest_concentration_row['Label']),
    'highest_rf_score_concentration_percentage':
        float(
            highest_concentration_row[
                'RF Most Frequent Score Share (%)'
            ]
        ),
    'dt_to_rf_training_time_ratio':
        float(training_speed_ratio),
    'prediction_parallel_jobs_per_forest': 1,
    'additional_hyperparameter_tuning_planned': False,
    'retained_as_nonlinear_ensemble_baseline': True,
    'probability_calibration_established': False,
    'statistical_significance_established': False,
    'individual_ensemble_mechanisms_isolated': False,
    'final_project_model_frozen': False,
    'official_test_used': False
}

RF_ANALYSIS_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'rf_baseline_analysis.json'
)

with open(RF_ANALYSIS_CONFIG_PATH, 'w') as file:

    json.dump(
        rf_analysis_config,
        file,
        indent=4
    )

with open(RF_ANALYSIS_CONFIG_PATH, 'r') as file:

    assert json.load(file) == rf_analysis_config


# Summarize completed comparative analysis

print("\nRandom Forest comparative analysis summary")

print("Common-feature models compared:",
      len(baseline_results))

print("Metric and prediction reconstruction: Passed")

print("Random Forest common-baseline rank:",
      rf_rank, "/", len(baseline_results))

print("Validation Macro AP:",
      round(run_summary['Macro AP'], 6))

print("Optimized Macro F1:",
      round(run_summary['Optimized Macro F1'], 6))

print("Macro AP improvement versus Decision Tree:",
      round(macro_ap_improvement, 6))

print("Optimized Macro F1 improvement versus Decision Tree:",
      round(macro_f1_improvement, 6))

print("Decision Tree Macro AP generalization gap:",
      round(dt_ap_gap, 6))

print("Random Forest Macro AP generalization gap:",
      round(rf_ap_gap, 6))

print("Relative AP gap reduction (%):",
      round(ap_gap_reduction_percentage, 4))

print("Labels with improved AP versus Decision Tree:",
      ap_improved_labels, "/", len(LABELS))

print("Labels with improved optimized F1 versus Decision Tree:",
      f1_improved_labels, "/", len(LABELS))

print("Labels with reduced AP gap:",
      gap_reduced_labels, "/", len(LABELS))

print("Labels with no default positive predictions:",
      zero_default_prediction_labels)

print("Largest Random Forest AP gap label:",
      largest_gap_row['Label'])

print("Highest Random Forest score concentration label:",
      highest_concentration_row['Label'])

print("Decision Tree to Random Forest training time ratio:",
      round(training_speed_ratio, 4))

print("Research figures generated: 2")
print("Analysis configuration: Saved and verified")
print("Additional model training: No")
print("Experiment registration: Pending")
print("Official test dataset: Not loaded")

gc.collect()

print("\nRandom Forest comparative analysis completed successfully.")

In [ ]:
# Verify completed experiment and evaluation status

assert len(rf_results) == 1
assert RUN_ID in rf_results
assert run_summary['Status'] == 'Completed'
assert run_summary['Structural Verification'] == 'Passed'
assert not run_summary['Official Test Used']
assert maximum_reconstruction_error <= 1e-12

with open(TRAINING_LOG_PATH, 'r') as file:
    saved_training_log = json.load(file)

required_checks = [
    'all_structural_checks_passed',
    'class_alignment_verified',
    'bootstrap_support_verified',
    'probability_averaging_verified',
    'score_alignment_verified',
    'model_reconstruction_verified',
    'metric_reconstruction_verified',
    'prediction_roundtrip_verified',
    'threshold_roundtrip_verified',
    'validation_evaluation_completed',
    'threshold_optimization_completed'
]

assert all(saved_training_log[key] for key in required_checks)
assert saved_training_log['status'] == 'Completed'
assert not saved_training_log['official_test_used']
assert saved_training_log['prediction_parallel_jobs_per_forest'] == 1


# Verify critical artifact fingerprints

saved_manifest_df = pd.read_csv(
    RUN_TABLES_DIR / 'artifact_manifest.csv'
)

assert len(saved_manifest_df) == 7
assert saved_manifest_df['Run ID'].eq(RUN_ID).all()
assert saved_manifest_df['Artifact Type'].is_unique

for _, artifact in saved_manifest_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()
    assert path.stat().st_size == artifact['Size Bytes']

    assert calculate_sha256(path) == artifact['SHA256'], (
        f"Fingerprint mismatch: {path}"
    )

verified_fingerprint_count = len(saved_manifest_df)

print("Critical artifact fingerprints verified:",
      verified_fingerprint_count)


# Verify all fitted trees

saved_tree_structure_df = pd.read_csv(
    RUN_TABLES_DIR / 'individual_tree_structure.csv'
)

saved_structure_df = pd.read_csv(
    RUN_TABLES_DIR / 'classifier_structure_summary.csv'
)

assert len(saved_tree_structure_df) == total_trees
assert len(saved_structure_df) == len(LABELS)

assert saved_structure_df['Label'].tolist() == LABELS

assert saved_tree_structure_df[
    'Structural Verification'
].eq('Passed').all()

assert saved_structure_df[
    'Class Alignment'
].eq('Passed').all()

assert saved_tree_structure_df[
    'Actual Depth'
].le(RF_PARAMS['max_depth']).all()

assert saved_tree_structure_df[
    'Leaf Count'
].le(maximum_leaves).all()

assert saved_tree_structure_df[
    'Minimum Leaf Samples'
].ge(RF_PARAMS['min_samples_leaf']).all()

assert saved_tree_structure_df[
    'Bootstrap Draws'
].eq(len(y_train)).all()

assert saved_tree_structure_df['Both Classes Present'].all()

assert (
    saved_tree_structure_df['Node Count']
    == 2 * saved_tree_structure_df['Leaf Count'] - 1
).all()

assert not saved_tree_structure_df.duplicated(
    ['Label', 'Tree Index']
).any()

assert saved_structure_df['Trees'].eq(
    trees_per_label
).all()

print("Individual tree verification:",
      len(saved_tree_structure_df), "/", total_trees)


# Define required notebook-level tables

required_tables = [
    'previous_run_consistency.csv',
    'ensemble_probability_evaluation_verification.csv',
    'environment_versions.csv',
    'environment_compatibility.csv',
    'dataset_verification.csv',
    'feature_representation_verification.csv',
    'previous_feature_consistency.csv',
    'label_support_and_imbalance.csv',
    'rf_bootstrap_support_estimates.csv',
    'dt_rf_parameter_comparison.csv',
    'rf_hyperparameter_rationale.csv',
    'rf_structural_limits.csv',
    'rf_rare_label_bootstrap_support.csv',
    'rf_experiment_plan.csv',
    'rf_common_baseline_comparison.csv',
    'rf_common_baseline_per_label.csv',
    'rf_common_baseline_ranking.csv',
    'rf_common_baseline_ap_comparison.csv',
    'rf_common_baseline_f1_comparison.csv',
    'rf_dt_overall_comparison.csv',
    'rf_dt_per_label_comparison.csv',
    'rf_dt_rare_label_comparison.csv',
    'rf_common_baseline_threshold_sensitivity.csv'
]

required_run_tables = [
    'individual_tree_structure.csv',
    'classifier_structure_summary.csv',
    'bootstrap_support_verification.csv',
    'model_complexity_summary.csv',
    'probability_averaging_verification.csv',
    'probability_score_distribution.csv',
    'overall_threshold_comparison.csv',
    'per_label_threshold_comparison.csv',
    'training_validation_comparison.csv',
    'per_label_generalization_comparison.csv',
    'default_overall_metrics.csv',
    'optimized_overall_metrics.csv',
    'training_default_overall_metrics.csv',
    'training_optimized_overall_metrics.csv',
    'default_per_label_metrics.csv',
    'optimized_per_label_metrics.csv',
    'training_default_per_label_metrics.csv',
    'training_optimized_per_label_metrics.csv',
    'optimized_thresholds.csv',
    'feature_verification.csv',
    'run_summary.csv',
    'artifact_manifest.csv'
]

required_figures = [
    'rf_common_baseline_threshold_sensitivity.png',
    'rf_dt_predictive_diagnostics.png'
]

required_configs = [
    'rf_experiment_config.json',
    'rf_baseline_analysis.json',
    f'{RUN_ID}.json',
    f'{RUN_ID}_thresholds.json'
]


# Build the final artifact inventory

verification_records = []

def verify_artifact(path, category):

    available = path.is_file() and path.stat().st_size > 0

    verification_records.append({
        'Type': category,
        'Filename': path.name,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'Size Bytes': path.stat().st_size if available else 0,
        'SHA256': calculate_sha256(path) if available else '',
        'Status': 'Available' if available else 'Missing'
    })


for folder, filenames, category in [
    (TABLES_DIR, required_tables, 'Analysis Table'),
    (RUN_TABLES_DIR, required_run_tables, 'Run Table'),
    (FIGURES_DIR, required_figures, 'Figure'),
    (RUN_CONFIGS_DIR, required_configs, 'Configuration'),
    (
        RUN_LOGS_DIR,
        [
            'training_log.json',
            'training_warnings.json',
            'training_memory_usage.json'
        ],
        'Training Log'
    ),
    (
        LOGS_DIR,
        [
            'runtime_resources.json',
            'loaded_runtime_resources.json'
        ],
        'Runtime Log'
    )
]:

    for filename in filenames:
        verify_artifact(folder / filename, category)

verify_artifact(MODEL_PATH, 'Model')
verify_artifact(PREDICTIONS_PATH, 'Prediction')


# Verify shared dependencies

word_vectorizer_path = (
    PROJECT_ROOT / '04_models' / 'classical'
    / 'vectorizers' / 'word_tfidf.joblib'
)

word_vectorizer_record = feature_manifest.loc[
    (feature_manifest['Artifact Type'] == 'Vectorizer')
    & (
        feature_manifest['Path']
        == str(word_vectorizer_path.relative_to(PROJECT_ROOT))
    )
]

assert len(word_vectorizer_record) == 1

assert calculate_sha256(word_vectorizer_path) == (
    word_vectorizer_record.iloc[0]['SHA256']
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash

for path in [
    word_vectorizer_path,
    SRC_DIR / 'evaluation_utils.py',
    FEATURES_DIR / 'feature_artifact_manifest.csv'
]:
    verify_artifact(path, 'Shared Dependency')


# Verify artifact completeness

final_verification_df = pd.DataFrame(
    verification_records
)

assert final_verification_df['Path'].is_unique

missing_artifacts_df = final_verification_df.loc[
    final_verification_df['Status'] == 'Missing'
]

if not missing_artifacts_df.empty:
    display(missing_artifacts_df)

assert missing_artifacts_df.empty

final_verification_df.to_csv(
    TABLES_DIR / 'rf_final_artifact_verification.csv',
    index=False
)

print("\nFinal artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)


# Verify the completed analysis configuration

with open(RF_ANALYSIS_CONFIG_PATH, 'r') as file:
    saved_analysis_config = json.load(file)

assert saved_analysis_config == rf_analysis_config
assert saved_analysis_config['common_baseline_rank'] == rf_rank
assert saved_analysis_config['retained_as_nonlinear_ensemble_baseline']
assert not saved_analysis_config['official_test_used']
assert not saved_analysis_config['final_project_model_frozen']


# Prepare the experiment registry record

registry_record = {
    'run_id': RUN_ID,
    'notebook': NOTEBOOK_DIR.name,
    'model': MODEL_NAME,
    'feature_representation': REPRESENTATION,
    'loss_function': 'Gini impurity (split criterion)',
    'sampling_strategy': 'representative_multilabel',
    'training_samples': len(y_train),
    'seed': SEED,
    'max_length': '',
    'config_path': str(
        RUN_CONFIG_PATH.relative_to(PROJECT_ROOT)
    ),
    'model_path': str(
        MODEL_PATH.relative_to(PROJECT_ROOT)
    ),
    'results_path': str(
        RUN_DIR.relative_to(PROJECT_ROOT)
    ),
    'validation_macro_ap': run_summary['Macro AP'],
    'validation_macro_f1': run_summary['Optimized Macro F1'],
    'training_time_seconds': training_time,
    'status': 'Completed',
    'start_time': training_start_time,
    'end_time': training_end_time,
    'notes': (
        'Resource-controlled unweighted Random Forest. '
        '50 trees per label, maximum depth 30, '
        'maximum 512 leaves, minimum leaf size 5. '
        'Full-size bootstrap and sqrt feature sampling. '
        'Two training jobs and sequential prediction. '
        'Training time includes CSC conversion. '
        'Validation-only per-label threshold optimization. '
        'One datetime deprecation warning retained. '
        'Official test not used.'
    )
}

registry_before_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry_before_df['run_id'].is_unique

assert set(registry_before_df.columns) == set(
    registry_record.keys()
)

previous_registry_df = registry_before_df.loc[
    registry_before_df['run_id'] != RUN_ID
].copy()

assert set(previous_registry_df['run_id']) == set(
    required_previous_runs
)

new_record_df = pd.DataFrame(
    [registry_record],
    columns=registry_before_df.columns
).astype(str)

existing_rf_record = registry_before_df.loc[
    registry_before_df['run_id'] == RUN_ID
]

if not existing_rf_record.empty:

    pd.testing.assert_frame_equal(
        existing_rf_record.reset_index(drop=True),
        new_record_df.reset_index(drop=True),
        check_exact=True
    )

updated_registry_df = pd.concat(
    [previous_registry_df, new_record_df],
    ignore_index=True
)

assert updated_registry_df['run_id'].is_unique
assert len(updated_registry_df) == 11


# Record methodological decisions

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert decisions_df['decision_id'].is_unique

rf_decisions = [
    {
        'decision_id': 'D043',
        'component': 'Random Forest configuration',
        'decision': (
            'Use 50 trees per label with full-size bootstrap '
            'sampling and sqrt feature selection'
        ),
        'justification': (
            'Retain the common word TF-IDF representation '
            'and Decision Tree structural restrictions. '
            'Limit ensemble size to preserve computational '
            'resources for mandatory transformer experiments.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D044',
        'component': 'Random Forest structural verification',
        'decision': (
            'Verify every individual tree and its actual '
            'bootstrap class support'
        ),
        'justification': (
            'All 300 trees satisfy the predefined limits '
            'and contain both classes. Observed bootstrap '
            'support is consistent with theoretical expectations. '
            'Corresponding trees across labels use shared '
            'bootstrap draws due to the common random seed.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D045',
        'component': 'Random Forest prediction reproducibility',
        'decision': (
            'Use sequential tree probability averaging '
            'during validation and subsequent inference'
        ),
        'justification': (
            'Retain parallel training while avoiding changes '
            'in floating-point accumulation order at threshold '
            'boundaries. Reloaded validation scores were '
            'reconstructed with zero error.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D046',
        'component': 'Random Forest generalization',
        'decision': (
            'Retain the ensemble comparison and '
            'label-specific limitations as research evidence'
        ),
        'justification': (
            f'Macro AP improves by {macro_ap_improvement:.6f} '
            'over Decision Tree. The overall AP generalization '
            f'gap decreases by {ap_gap_reduction_percentage:.4f}%. '
            'Threat remains an exception with a larger AP gap '
            'and lower optimized F1.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D047',
        'component': 'Random Forest experiment scope',
        'decision': (
            'Retain one predefined Random Forest baseline '
            'without additional hyperparameter tuning'
        ),
        'justification': (
            f'The configuration ranks {rf_rank} of '
            f'{len(baseline_results)} common-feature baselines. '
            f"Validation Macro F1 improves from "
            f"{default_metrics['Macro F1']:.6f} to "
            f"{optimized_metrics['Macro F1']:.6f} after tuning. "
            'Further forest optimization is deprioritized '
            'in favor of mandatory project deliverables.'
        ),
        'status': 'Implemented'
    }
]

assert set(decisions_df.columns) == set(
    rf_decisions[0].keys()
)

rf_decision_ids = [
    record['decision_id']
    for record in rf_decisions
]

previous_decisions_df = decisions_df.loc[
    ~decisions_df['decision_id'].isin(rf_decision_ids)
].copy()

assert set(previous_decisions_df['decision_id']) == {
    f'D{i:03d}' for i in range(1, 43)
}

new_decisions_df = pd.DataFrame(
    rf_decisions,
    columns=decisions_df.columns
)

for _, record in new_decisions_df.iterrows():

    existing = decisions_df.loc[
        decisions_df['decision_id'] == record['decision_id']
    ]

    if not existing.empty:

        pd.testing.assert_frame_equal(
            existing.reset_index(drop=True),
            record.to_frame().T.reset_index(drop=True),
            check_dtype=False
        )

updated_decisions_df = pd.concat(
    [previous_decisions_df, new_decisions_df],
    ignore_index=True
)

assert updated_decisions_df['decision_id'].is_unique
assert len(updated_decisions_df) == 47


# Verify and save shared CSV updates

def save_verified_csv(dataframe, path):

    temporary_path = path.with_suffix('.tmp')

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    reloaded = pd.read_csv(
        temporary_path,
        dtype=str,
        keep_default_na=False
    )

    pd.testing.assert_frame_equal(
        dataframe.astype(str).reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_exact=True
    )

    temporary_path.replace(path)

    return pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False
    )


saved_registry_df = save_verified_csv(
    updated_registry_df,
    REGISTRY_PATH
)

saved_decisions_df = save_verified_csv(
    updated_decisions_df,
    DECISION_LOG_PATH
)

pd.testing.assert_frame_equal(
    previous_registry_df.reset_index(drop=True),
    saved_registry_df.loc[
        saved_registry_df['run_id'] != RUN_ID
    ].reset_index(drop=True),
    check_exact=True
)

pd.testing.assert_frame_equal(
    previous_decisions_df.reset_index(drop=True),
    saved_decisions_df.loc[
        ~saved_decisions_df['decision_id'].isin(rf_decision_ids)
    ].reset_index(drop=True),
    check_exact=True
)

rf_registered_record = saved_registry_df.loc[
    saved_registry_df['run_id'] == RUN_ID
].iloc[0]

assert np.isclose(
    float(rf_registered_record['validation_macro_ap']),
    run_summary['Macro AP'],
    rtol=0,
    atol=1e-12
)

assert np.isclose(
    float(rf_registered_record['validation_macro_f1']),
    run_summary['Optimized Macro F1'],
    rtol=0,
    atol=1e-12
)


# Generate the complete classical validation leaderboard

leaderboard_df = saved_registry_df.loc[
    saved_registry_df['status'] == 'Completed',
    [
        'run_id',
        'model',
        'feature_representation',
        'training_samples',
        'validation_macro_ap',
        'validation_macro_f1',
        'training_time_seconds',
        'status'
    ]
].copy()

for column in [
    'training_samples',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]:

    leaderboard_df[column] = pd.to_numeric(
        leaderboard_df[column]
    )

leaderboard_df = (
    leaderboard_df.sort_values(
        [
            'validation_macro_ap',
            'validation_macro_f1',
            'run_id'
        ],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

leaderboard_df.insert(
    0, 'Rank',
    np.arange(1, len(leaderboard_df) + 1)
)

assert len(leaderboard_df) == 11

leaderboard_df.to_csv(
    TABLES_DIR / 'rf_current_validation_leaderboard.csv',
    index=False
)

print("\nUpdated classical validation leaderboard")

display(leaderboard_df.round(6))

print("\nNew methodological decisions")

display(new_decisions_df)


# Build the research summary table

summary_rows = [
    '| Model | Macro AP | Macro ROC-AUC | Optimized Macro F1 | Optimized Micro F1 |',
    '|---|---:|---:|---:|---:|'
]

for _, row in rf_baseline_ranking_df.iterrows():

    summary_rows.append(
        f"| {row['Model']} "
        f"| {row['Macro AP']:.6f} "
        f"| {row['Macro ROC-AUC']:.6f} "
        f"| {row['Optimized Macro F1']:.6f} "
        f"| {row['Optimized Micro F1']:.6f} |"
    )

summary_table_text = '\n'.join(summary_rows)


# Save the final Random Forest research summary

summary = f"""# Notebook 03.05 - Random Forest

## Objective

Evaluate a resource-controlled nonlinear ensemble using
the common word TF-IDF features and identical development
partitions used by previous classical baselines.

## Experimental Configuration

- Run ID: {RUN_ID}
- Classification strategy: One-vs-Rest
- Training comments: {len(y_train):,}
- Validation comments: {len(y_val):,}
- Features: {FEATURE_BUDGET:,}
- Binary forests: {len(LABELS)}
- Trees per forest: {trees_per_label}
- Total trees: {total_trees}
- Criterion: Gini
- Maximum depth: {RF_PARAMS['max_depth']}
- Maximum leaves per tree: {maximum_leaves}
- Minimum leaf size: {RF_PARAMS['min_samples_leaf']}
- Feature sampling: sqrt
- Nominal candidate features: {candidate_features_per_split}
- Bootstrap draws per tree: {len(y_train):,}
- Class weighting: None
- Random seed: {SEED}
- Training jobs: {RF_PARAMS['n_jobs']}
- Prediction jobs per forest: 1

## Common Word TF-IDF Comparison

{summary_table_text}

Random Forest ranks {rf_rank} of {len(baseline_results)}
common-feature configurations by validation Macro AP.

## Decision Tree versus Random Forest

- Macro AP improvement: {macro_ap_improvement:.6f}
- Optimized Macro F1 improvement: {macro_f1_improvement:.6f}
- Decision Tree Macro AP gap: {dt_ap_gap:.6f}
- Random Forest Macro AP gap: {rf_ap_gap:.6f}
- Relative AP gap reduction: {ap_gap_reduction_percentage:.4f}%
- Labels with improved AP: {ap_improved_labels}/6
- Labels with improved optimized F1: {f1_improved_labels}/6
- Labels with reduced AP gap: {gap_reduced_labels}/6

The evaluated Random Forest improves overall validation
ranking and classification performance relative to Decision
Tree. However, threat exhibits a larger generalization gap
and lower optimized F1.

The comparison does not isolate the individual contributions
of bootstrap sampling, randomized feature selection or
probability averaging.

## Threshold Optimization

- Default Macro F1: {default_metrics['Macro F1']:.6f}
- Optimized Macro F1: {optimized_metrics['Macro F1']:.6f}
- Default Micro F1: {default_metrics['Micro F1']:.6f}
- Optimized Micro F1: {optimized_metrics['Micro F1']:.6f}
- Labels without default positive predictions:
  {', '.join(zero_default_prediction_labels)}

Validation-only per-label threshold optimization substantially
improves classification performance. Ranking metrics remain
unchanged because threshold selection does not modify scores.

## Probability Resolution

The ensemble produces more distinct validation probability
scores than the evaluated individual Decision Tree.

The largest remaining score concentration occurs for
{highest_concentration_row['Label']} at
{highest_concentration_row['RF Most Frequent Score Share (%)']:.6f}%.

Distinct-score counts are based on exact floating-point values
and do not establish probability calibration.

## Structural and Computational Findings

- Verified individual trees: {len(saved_tree_structure_df)}
- Total leaves: {saved_tree_structure_df['Leaf Count'].sum()}
- Total nodes: {saved_tree_structure_df['Node Count'].sum()}
- Trees missing a class: {saved_structure_df['Trees Missing a Class'].sum()}
- Model fitting time: {model_fit_time:.6f} seconds
- Total training time: {training_time:.6f} seconds
- Validation prediction time: {validation_prediction_time:.6f} seconds
- Saved model size: {model_size_mib:.6f} MiB
- Sampled peak process memory: {sampled_peak_memory_mib:.6f} MiB
- Sampled peak memory increase:
  {memory_summary['sampled_peak_increase_mib']:.6f} MiB

The recorded training time is lower than Decision Tree by
a factor of {training_speed_ratio:.4f}.

Runtime comparisons are descriptive. The algorithms use
different feature-search strategies and parallelization.
Prediction uses sequential averaging for reproducibility.

## Reproducibility

- Critical artifact fingerprints verified: {verified_fingerprint_count}
- Required artifact entries: {len(final_verification_df)}
- Missing artifacts: {len(missing_artifacts_df)}
- Maximum score reconstruction error: {maximum_reconstruction_error}
- Maximum probability averaging error: {max(averaging_errors)}
- Threshold and prediction roundtrips: Passed
- Model and metric reconstruction: Passed
- Experiment registration: Completed

One datetime deprecation warning is retained in the logs.

## Limitations

- Only one predefined Random Forest configuration evaluated.
- No hyperparameter optimization performed.
- Single validation split and random seed.
- Validation-selected threshold performance may be optimistic.
- Ensemble probabilities are not necessarily calibrated.
- Statistical significance has not been established.
- Memory usage is sampled process RSS, not exact peak memory.
- Saved model size excludes the shared vectorizer.
- Corresponding trees across labels share bootstrap draws.

## Research Decision

Retain Random Forest as the planned nonlinear ensemble baseline.
Do not conduct additional Random Forest training at this stage.

The current leading registered classical configuration is
{leaderboard_df.loc[0, 'run_id']}, ranked by validation Macro AP.

The final classical selection will be consolidated in Notebook 03.06.
The final project model has not been frozen.

Official test data has not been used for model training,
threshold optimization or model selection.

## Next Stage

Notebook 03.06 - Consolidated Classical ML Comparison.
"""

summary_path = RESULTS_DIR / 'summary.md'

summary_path.write_text(
    summary,
    encoding='utf-8'
)

assert summary_path.read_text(
    encoding='utf-8'
) == summary


# Summarize notebook completion

print("\nRandom Forest completion summary")

print("Notebook-level analysis tables:", len(required_tables))
print("Individual experiment tables:", len(required_run_tables))
print("Research figures:", len(required_figures))

print("Completed Random Forest experiments:", len(rf_results))
print("Verified binary forests:", len(saved_structure_df))
print("Verified individual trees:", len(saved_tree_structure_df))

print("Critical fingerprint checks:", verified_fingerprint_count)
print("Total required artifact entries:", len(final_verification_df))
print("Missing artifacts:", len(missing_artifacts_df))

print("Previously registered experiments:", len(previous_registry_df))
print("Total registered experiments:", len(saved_registry_df))

print("Duplicate run IDs:",
      saved_registry_df['run_id'].duplicated().sum())

print("Total methodological decisions:", len(saved_decisions_df))

print("Random Forest baseline rank:",
      rf_rank, "/", len(baseline_results))

print("Current leading configuration:",
      leaderboard_df.loc[0, 'run_id'])

print("Previous registry records: Preserved")
print("Previous methodological decisions: Preserved")
print("Notebook summary saved.")
print("Official test dataset: Not used")
print("Final project model frozen: No")

print("\nRandom Forest notebook completed successfully.")